In [1]:
"""
═══════════════════════════════════════════════════════════
Central configuration, all typed data structures, granularity enums,
and terminal logger for the REFRAG system.

REFRAG Novelty:
  Standard RAG retrieves whole chunks (512–1024 tokens) and returns them.
  REFRAG retrieves at three sub-document granularities simultaneously:
    PROPOSITION  — atomic single-claim units (~10–40 words)
    SENTENCE     — full sentence spans (~20–80 words)
    CHUNK        — paragraph-level context (~300–700 words)

  When coverage of query sub-topics is incomplete, REFRAG iteratively
  re-retrieves with targeted follow-up queries until coverage ≥ threshold
  or max iterations reached.

  Final answers cite at proposition-level precision — every claim in
  the answer maps to an exact indexed proposition.

References:
  • Dense Passage Retrieval (DPR) — Karpukhin et al. (2020)
    https://arxiv.org/abs/2004.04906
  • FLARE: Forward-Looking Active Retrieval — Jiang et al. (2023)
    https://arxiv.org/abs/2305.06983
  • Proposition-based RAG (Chen et al., 2023)
    https://arxiv.org/abs/2312.06648
  • RAPTOR: Recursive Abstractive Processing — Sarthi et al. (2024)
    https://arxiv.org/abs/2401.18059
  • Sub-question Decomposition (Perez et al., 2020)
    https://arxiv.org/abs/2002.09758
  • Hypothetical Document Embedding (HyDE) — Gao et al. (2022)
    https://arxiv.org/abs/2212.10496
  • LLM-based Re-ranking (Ma et al., 2023)
    https://arxiv.org/abs/2310.07554
  • Self-RAG — Asai et al. (2023)
    https://arxiv.org/abs/2310.11511
"""

'\n═══════════════════════════════════════════════════════════\nCentral configuration, all typed data structures, granularity enums,\nand terminal logger for the REFRAG system.\n\nREFRAG Novelty:\n  Standard RAG retrieves whole chunks (512–1024 tokens) and returns them.\n  REFRAG retrieves at three sub-document granularities simultaneously:\n    PROPOSITION  — atomic single-claim units (~10–40 words)\n    SENTENCE     — full sentence spans (~20–80 words)\n    CHUNK        — paragraph-level context (~300–700 words)\n\n  When coverage of query sub-topics is incomplete, REFRAG iteratively\n  re-retrieves with targeted follow-up queries until coverage ≥ threshold\n  or max iterations reached.\n\n  Final answers cite at proposition-level precision — every claim in\n  the answer maps to an exact indexed proposition.\n\nReferences:\n  • Dense Passage Retrieval (DPR) — Karpukhin et al. (2020)\n    https://arxiv.org/abs/2004.04906\n  • FLARE: Forward-Looking Active Retrieval — Jiang et al. (202

In [32]:
from __future__ import annotations
from langchain_core.messages import HumanMessage
import chromadb
from chromadb.config import Settings
import os
import time
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple
from dataclasses import dataclass, field


In [3]:

# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    # ── Azure OpenAI (AI Foundry) ─────────────────────────────────────
    AI_FOUNDRY_PROJECT_ENDPOINT: str = field(default_factory=lambda: os.getenv(
        "AI_FOUNDRY_PROJECT_ENDPOINT", "https://idkopenai.services.ai.azure.com"))
    AI_FOUNDRY_DEPLOYMENT_NAME: str = field(default_factory=lambda: os.getenv(
        "AI_FOUNDRY_DEPLOYMENT_NAME", "gpt-4o"))
    AI_FOUNDRY_API_VERSION: str = field(default_factory=lambda: os.getenv(
        "AI_FOUNDRY_API_VERSION", "2024-12-01-preview"))
    AI_FOUNDRY_API_KEY: str = field(default_factory=lambda: os.getenv(
        "AI_FOUNDRY_API_KEY", os.getenv("AZURE_OPENAI_API_KEY", "")))

    # ── HuggingFace Embeddings (local) ───────────────────────────────
    EMBEDDING_MODEL: str  = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str = field(default_factory=lambda: os.getenv("EMBEDDING_DEVICE", "cpu"))

    # ── Storage ───────────────────────────────────────────────────────
    BASE_DIR: str              = "./refrag_store"
    CHROMA_DIR: str            = "./refrag_store/chroma"
    PROP_COLLECTION: str       = "refrag_propositions"
    SENT_COLLECTION: str       = "refrag_sentences"
    CHUNK_COLLECTION: str      = "refrag_chunks"

    # ── Chunking / Splitting ──────────────────────────────────────────
    CHUNK_SIZE: int            = 600    # characters per chunk
    CHUNK_OVERLAP: int         = 100
    MIN_SENTENCE_LEN: int      = 20     # chars; shorter spans merged
    MIN_PROP_LEN: int          = 15     # chars; shorter propositions dropped

    # ── Retrieval ─────────────────────────────────────────────────────
    TOP_K_PROP: int            = 8      # propositions per sub-query
    TOP_K_SENT: int            = 5      # sentences per sub-query
    TOP_K_CHUNK: int           = 4      # chunks per sub-query
    FETCH_K: int               = 16     # MMR fetch pool
    MMR_LAMBDA: float          = 0.65   # diversity vs relevance

    # ── Re-retrieval loop ─────────────────────────────────────────────
    MAX_RETRIEVAL_ROUNDS: int  = 3      # max iterative re-retrieval rounds
    COVERAGE_THRESHOLD: float  = 0.75   # stop re-retrieving when coverage ≥ this
    MIN_COVERAGE_GAIN: float   = 0.05   # stop if coverage gain < this per round

    # ── Query decomposition ───────────────────────────────────────────
    MAX_SUB_QUESTIONS: int     = 4      # max sub-questions per query

    # ── Citation precision ────────────────────────────────────────────
    # If proposition similarity to an answer sentence ≥ this, cite it
    CITATION_SIM_THRESHOLD: float = 0.70

    # ── Re-ranking ────────────────────────────────────────────────────
    RERANK_TOP_N: int          = 12     # candidates going into LLM re-ranker
    RERANK_FINAL_K: int        = 6      # final passages after re-ranking

    # ── LLM ──────────────────────────────────────────────────────────
    LLM_TEMPERATURE: float     = 0.0
    LLM_MAX_TOKENS: int        = 1400

    def is_configured(self) -> bool:
        """Check Azure OpenAI credentials are present."""
        return bool(
            self.AI_FOUNDRY_PROJECT_ENDPOINT
            and self.AI_FOUNDRY_DEPLOYMENT_NAME
            and self.AI_FOUNDRY_API_VERSION
            and self.AI_FOUNDRY_API_KEY
        )

In [4]:
# ══════════════════════════════════════════════════════════════════════
# GRANULARITY ENUM
# ══════════════════════════════════════════════════════════════════════

class Granularity(str, Enum):
    PROPOSITION = "proposition"   # atomic single-claim (~10–40 words)
    SENTENCE    = "sentence"      # full sentence (~20–80 words)
    CHUNK       = "chunk"         # paragraph context (~300–700 words)


class RetrievalStrategy(str, Enum):
    SINGLE_SHOT     = "single_shot"     # one retrieval, no re-retrieval
    ITERATIVE       = "iterative"       # re-retrieve until coverage threshold
    DECOMPOSED      = "decomposed"      # query → sub-questions → fuse
    HYDE            = "hyde"            # hypothetical document embedding
    FULL_REFRAG     = "full_refrag"     # decompose + multi-granularity + iterative


class CoverageStatus(str, Enum):
    FULL      = "full"       # coverage ≥ threshold
    PARTIAL   = "partial"    # some sub-topics covered
    SPARSE    = "sparse"     # < 30% coverage
    NONE      = "none"       # no relevant results



In [5]:

# ══════════════════════════════════════════════════════════════════════
# FINE-GRAINED UNIT TYPES
# ══════════════════════════════════════════════════════════════════════

@dataclass
class Proposition:
    """
    An atomic, independently verifiable claim extracted from a document.
    The finest retrieval unit in REFRAG.

    Based on: "Propositional Content in Retrieval-Augmented Generation"
    (Chen et al., 2023) https://arxiv.org/abs/2312.06648
    """
    prop_id:      str
    content:      str          # the atomic claim text
    doc_id:       str          # source document ID
    chunk_id:     str          # parent chunk ID
    sent_id:      str          # parent sentence ID
    source:       str          # document source label
    page_num:     Optional[int] = None
    position:     int          = 0     # position within parent sentence
    confidence:   float        = 1.0   # extraction confidence (0–1)
    entity_spans: List[str]    = field(default_factory=list)  # key entities mentioned
    metadata:     Dict[str, Any] = field(default_factory=dict)

    def to_citation(self) -> str:
        loc = f"p.{self.page_num}" if self.page_num else f"§{self.position}"
        return f"[{self.source} · {loc} · prop]"

    def __len__(self) -> int:
        return len(self.content.split())


@dataclass
class SentenceUnit:
    """
    A full sentence span — middle granularity.
    Contains multiple propositions; provides fuller context than a proposition alone.
    """
    sent_id:     str
    content:     str
    doc_id:      str
    chunk_id:    str
    source:      str
    page_num:    Optional[int] = None
    position:    int           = 0      # sentence index within chunk
    prop_ids:    List[str]     = field(default_factory=list)  # child proposition IDs
    metadata:    Dict[str, Any] = field(default_factory=dict)

    def to_citation(self) -> str:
        loc = f"p.{self.page_num}" if self.page_num else f"§{self.position}"
        return f"[{self.source} · {loc} · sent]"


@dataclass
class ChunkUnit:
    """
    A paragraph-level chunk — coarsest retrieval unit.
    Provides full context; may contain multiple sentences and propositions.
    """
    chunk_id:   str
    content:    str
    doc_id:     str
    source:     str
    page_num:   Optional[int] = None
    sent_ids:   List[str]     = field(default_factory=list)
    prop_ids:   List[str]     = field(default_factory=list)
    metadata:   Dict[str, Any] = field(default_factory=dict)

    def to_citation(self) -> str:
        loc = f"p.{self.page_num}" if self.page_num else "§chunk"
        return f"[{self.source} · {loc} · chunk]"


@dataclass
class SubQuestion:
    """A decomposed sub-question derived from the main query."""
    sq_id:          str
    question:       str
    parent_query:   str
    focus:          str          # what aspect this sub-question targets
    granularity_hint: str        # which granularity is most appropriate
    answered:       bool         = False
    answer_text:    str          = ""
    coverage_score: float        = 0.0


@dataclass
class GranularityResult:
    """Retrieved results at one granularity level."""
    granularity:    Granularity
    items:          List[Any]          # Proposition | SentenceUnit | ChunkUnit
    scores:         Dict[str, float]   = field(default_factory=dict)
    sub_query:      str                = ""
    round_num:      int                = 1


@dataclass
class REFRAGResult:
    """
    Full result from a REFRAG query.
    Contains fine-grained retrievals across all granularities,
    coverage metrics, re-retrieval history, and cited answer.
    """
    query:              str
    strategy:           RetrievalStrategy
    sub_questions:      List[SubQuestion]          = field(default_factory=list)
    prop_results:       List[GranularityResult]    = field(default_factory=list)
    sent_results:       List[GranularityResult]    = field(default_factory=list)
    chunk_results:      List[GranularityResult]    = field(default_factory=list)
    reranked:           List[Any]                  = field(default_factory=list)
    answer:             str                        = ""
    citations:          List[Tuple[str, str]]      = field(default_factory=list)  # (claim, citation)
    coverage_history:   List[float]                = field(default_factory=list)
    final_coverage:     float                      = 0.0
    coverage_status:    str                        = CoverageStatus.NONE.value
    retrieval_rounds:   int                        = 0
    total_props:        int                        = 0
    total_sents:        int                        = 0
    total_chunks:       int                        = 0
    elapsed_sec:        float                      = 0.0

    def unique_sources(self) -> List[str]:
        sources = set()
        for gr in self.prop_results + self.sent_results + self.chunk_results:
            for item in gr.items:
                if hasattr(item, "source"):
                    sources.add(item.source)
        return sorted(sources)


In [6]:

# ══════════════════════════════════════════════════════════════════════
# TERMINAL LOGGER
# ══════════════════════════════════════════════════════════════════════

class C:
    RESET   = "\033[0m"; BOLD   = "\033[1m"; DIM    = "\033[2m"
    CYAN    = "\033[96m"; GREEN = "\033[92m"; YELLOW = "\033[93m"
    RED     = "\033[91m"; MAG   = "\033[95m"; BLUE   = "\033[94m"
    WHITE   = "\033[97m"; ORANGE= "\033[33m"
    PROP    = "\033[95m"   # magenta — proposition level
    SENT    = "\033[92m"   # green   — sentence level
    CHUNK   = "\033[96m"   # cyan    — chunk level
    RERANK  = "\033[93m"   # yellow  — re-ranking
    COVER   = "\033[94m"   # blue    — coverage
    DECOMP  = "\033[33m"   # orange  — decomposition
    HYDE    = "\033[91m"   # red     — HyDE


W = 76

GRAN_COLOURS = {
    Granularity.PROPOSITION: C.PROP,
    Granularity.SENTENCE:    C.SENT,
    Granularity.CHUNK:       C.CHUNK,
}


class Log:
    @staticmethod
    def banner(title: str, sub: str = ""):
        print(f"\n{C.BOLD}{C.WHITE}{'═'*W}")
        print(f"  {title}")
        if sub:
            print(f"  {C.DIM}{sub}{C.RESET}{C.BOLD}{C.WHITE}")
        print(f"{'═'*W}{C.RESET}")

    @staticmethod
    def section(title: str, colour: str = C.CYAN):
        print(f"\n{C.BOLD}{colour}{'━'*W}")
        print(f"  {title}")
        print(f"{'━'*W}{C.RESET}")

    @staticmethod
    def gran(gran: Granularity, msg: str):
        col = GRAN_COLOURS.get(gran, C.WHITE)
        tag = f"[{gran.value.upper():12s}]"
        print(f"{C.BOLD}{col}{tag}{C.RESET} {msg}")

    @staticmethod
    def step(msg: str):
        print(f"\n{C.BOLD}{C.BLUE}▶ {msg}{C.RESET}")

    @staticmethod
    def ok(msg: str):
        print(f"{C.GREEN}  ✓ {msg}{C.RESET}")

    @staticmethod
    def warn(msg: str):
        print(f"{C.YELLOW}  ⚠  {msg}{C.RESET}")

    @staticmethod
    def err(msg: str):
        print(f"{C.RED}  ✗ {msg}{C.RESET}")

    @staticmethod
    def info(msg: str):
        print(f"{C.DIM}  · {msg}{C.RESET}")

    @staticmethod
    def coverage(score: float, round_n: int, status: str):
        bar_len  = 30
        filled   = int(bar_len * score)
        bar      = "█" * filled + "░" * (bar_len - filled)
        colour   = C.GREEN if score >= 0.75 else (C.YELLOW if score >= 0.4 else C.RED)
        print(f"\n  {C.COVER}Coverage [round {round_n}]: "
              f"{colour}{bar}{C.RESET} {score:.1%} — {status}")

    @staticmethod
    def prop(p: "Proposition", score: float = 0.0):
        print(f"  {C.DIM}[{p.prop_id[:10]}]{C.RESET} "
              f"{C.PROP}{p.content[:80]}{C.RESET} "
              f"{C.DIM}score={score:.3f} src={p.source[:20]}{C.RESET}")

    @staticmethod
    def subq(sq: "SubQuestion", n: int):
        print(f"  {C.DECOMP}SQ{n}: {sq.question[:70]}{C.RESET}  "
              f"{C.DIM}[focus={sq.focus}, gran={sq.granularity_hint}]{C.RESET}")

    @staticmethod
    def answer_box(result: "REFRAGResult"):
        import textwrap
        print(f"\n{C.BOLD}{C.GREEN}{'═'*W}")
        print(f"  REFRAG ANSWER — Fine-Grained Retrieval-Enhanced RAG")
        print(f"{'═'*W}{C.RESET}")
        print(f"{C.BOLD}  Q: {result.query}{C.RESET}\n")
        for line in textwrap.wrap(result.answer, W - 4):
            print(f"  {line}")
        print(f"\n{C.DIM}{'─'*W}")
        print(f"  Strategy          : {result.strategy.value}")
        print(f"  Sub-questions     : {len(result.sub_questions)}")
        print(f"  Retrieval rounds  : {result.retrieval_rounds}")
        print(f"  Propositions used : {result.total_props}")
        print(f"  Sentences used    : {result.total_sents}")
        print(f"  Chunks used       : {result.total_chunks}")
        print(f"  Final coverage    : {result.final_coverage:.1%} ({result.coverage_status})")
        print(f"  Citations         : {len(result.citations)}")
        print(f"  Sources           : {result.unique_sources()}")
        print(f"  Coverage history  : {' → '.join(f'{s:.0%}' for s in result.coverage_history)}")
        print(f"  Elapsed           : {result.elapsed_sec:.2f}s")
        print(f"{'─'*W}{C.RESET}")


In [7]:
"""
══════════════════════════════════════════════════════════
Two corpora:

  KNOWLEDGE_BASE (10 reference papers on RAG / NLP / retrieval)
    — Used for KB-level retrieval in demos
    — All papers publicly available on arXiv

  DEMO_DOCUMENTS (3 long-form technical documents)
    — Rich, paragraph-length content
    — Used to demonstrate proposition extraction + fine-grained indexing
    — Each document has enough claims to show multi-granularity retrieval

Primary PDF to download:
  Dense Passage Retrieval (DPR) — Karpukhin et al. (2020)
  https://arxiv.org/pdf/2004.04906

📄 Reference papers:
  1.  DPR           https://arxiv.org/abs/2004.04906
  2.  FLARE         https://arxiv.org/abs/2305.06983
  3.  Propositionizer https://arxiv.org/abs/2312.06648
  4.  RAPTOR        https://arxiv.org/abs/2401.18059
  5.  Sub-Qa        https://arxiv.org/abs/2002.09758
  6.  HyDE          https://arxiv.org/abs/2212.10496
  7.  LLM Re-ranker https://arxiv.org/abs/2310.07554
  8.  Self-RAG      https://arxiv.org/abs/2310.11511
  9.  ColBERT       https://arxiv.org/abs/2004.12832
  10. FiD           https://arxiv.org/abs/2007.01282
"""

'\n══════════════════════════════════════════════════════════\nTwo corpora:\n\n  KNOWLEDGE_BASE (10 reference papers on RAG / NLP / retrieval)\n    — Used for KB-level retrieval in demos\n    — All papers publicly available on arXiv\n\n  DEMO_DOCUMENTS (3 long-form technical documents)\n    — Rich, paragraph-length content\n    — Used to demonstrate proposition extraction + fine-grained indexing\n    — Each document has enough claims to show multi-granularity retrieval\n\nPrimary PDF to download:\n  Dense Passage Retrieval (DPR) — Karpukhin et al. (2020)\n  https://arxiv.org/pdf/2004.04906\n\n📄 Reference papers:\n  1.  DPR           https://arxiv.org/abs/2004.04906\n  2.  FLARE         https://arxiv.org/abs/2305.06983\n  3.  Propositionizer https://arxiv.org/abs/2312.06648\n  4.  RAPTOR        https://arxiv.org/abs/2401.18059\n  5.  Sub-Qa        https://arxiv.org/abs/2002.09758\n  6.  HyDE          https://arxiv.org/abs/2212.10496\n  7.  LLM Re-ranker https://arxiv.org/abs/2310.07554\

In [8]:

# ══════════════════════════════════════════════════════════════════════
# KNOWLEDGE BASE — 10 reference papers
# ══════════════════════════════════════════════════════════════════════

KB_DOCS: List[Dict[str, str]] = [
    {
        "id": "dpr",
        "source": "Dense Passage Retrieval (DPR) — Karpukhin et al. (2020)",
        "url": "https://arxiv.org/abs/2004.04906",
        "content": """
Dense Passage Retrieval (DPR) uses dual BERT encoders to embed questions and passages
into a dense vector space, enabling efficient Maximum Inner Product Search (MIPS) with
FAISS. DPR replaces sparse BM25 retrieval with dense vectors that capture semantic meaning
beyond lexical overlap.

Architecture: two independent BERT-base encoders — one for questions, one for passages.
Training: contrastive learning with in-batch negatives; gold passages pushed closer to
questions, hard negatives pushed away. Tested against BM25 on open-domain QA:
NaturalQuestions top-20: DPR 79.4% vs BM25 59.1%. TriviaQA top-20: DPR 78.9% vs BM25 66.9%.
WebQuestions top-20: DPR 73.2% vs BM25 55.0%.

DPR uses a FAISS IVF index over 21 million Wikipedia 100-word passages.
Offline index construction: ~70 GPU-hours. Online retrieval: <1ms per query.
The DPR retriever is the retrieval backbone for the original RAG paper (Lewis et al., 2020).
Dense retrieval becomes progressively better than sparse as question complexity increases.
"""
    },
    {
        "id": "flare",
        "source": "FLARE: Forward-Looking Active Retrieval — Jiang et al. (2023)",
        "url": "https://arxiv.org/abs/2305.06983",
        "content": """
FLARE monitors token-level generation confidence and retrieves when uncertainty exceeds
a threshold δ (typically 0.4). This enables on-demand, fine-grained retrieval during
generation — unlike standard RAG which retrieves once at the start.

Algorithm: for each sentence being generated, if any token probability < δ, the
tentative sentence tokens (with high-probability tokens unmasked) form the retrieval query.
The model then retrieves relevant passages and regenerates the sentence using the new context.

This fine-grained retrieval pattern aligns with REFRAG's philosophy: retrieve at the
granularity you need, exactly when you need it.

FLARE benchmark results:
  StrategyQA: 60.1% (FLARE) vs 57.8% (standard RAG)
  ASQA EM-recall: 43.1% vs 39.7%
  2WikiMultihopQA: 46.2% vs 37.9%
  IIRC: 44.6% vs 38.2%

Key insight: fine-grained, span-level retrieval consistently outperforms coarse
one-shot retrieval, especially on multi-hop and long-form generation tasks.
"""
    },
    {
        "id": "propositionizer",
        "source": "Propositionizer: Dense Retrieval with Atomic Claims — Chen et al. (2023)",
        "url": "https://arxiv.org/abs/2312.06648",
        "content": """
The Propositionizer system decomposes passages into atomic propositions — minimal, self-
contained factual claims. Each proposition is independently indexed and retrieved,
enabling citation at the claim level rather than the passage level.

Proposition properties (three criteria):
  1. Minimal: contains exactly one factual claim
  2. Self-contained: interpretable without surrounding context
  3. Attributable: traceable back to a specific source span

Extraction process: given a passage, the model generates propositions in the form
"The [entity] [relation] [value]" or declarative sentences. Average proposition
length: 15–35 words vs 100-word passages.

Experiments on ALCE (Attributed LLM Evaluation):
  Citation precision: Propositionizer 71.4% vs chunk-level 48.2%
  Citation recall: 66.8% vs 52.1%
  Answer correctness: 58.9% vs 54.3%

The proposition-level index enables much more precise attribution than passage-level
retrieval. When a user asks "What is X's value for metric Y?", proposition retrieval
can return the exact sentence rather than a 100-word paragraph surrounding it.
"""
    },
    {
        "id": "raptor",
        "source": "RAPTOR: Recursive Abstractive Processing for Tree Organized Retrieval — Sarthi et al. (2024)",
        "url": "https://arxiv.org/abs/2401.18059",
        "content": """
RAPTOR builds a hierarchical tree of summaries over a document corpus. Leaf nodes
contain original text chunks; internal nodes contain LLM-generated summaries of their
children. Retrieval queries can be answered at the appropriate level of abstraction.

Algorithm:
  1. Cluster leaf chunks using Gaussian Mixture Models on embeddings
  2. Summarize each cluster with an LLM into a parent node
  3. Repeat recursively until a single root node covers the entire corpus
  4. At query time, search across ALL tree levels simultaneously
  5. Return the mixture of specific (leaf) and abstract (internal) nodes

RAPTOR complements REFRAG's multi-granularity approach: REFRAG goes finer
(propositions → sentences → chunks), RAPTOR goes coarser (chunks → summaries → root).
Combining both creates a full spectrum from atomic claims to document-level abstractions.

QuALITY benchmark (multi-hop comprehension): RAPTOR 82.6% vs flat RAG 73.2%.
NarrativeQA: 55.1% vs 41.2%. SummScreenFD: 44.2% vs 34.8%.
Tree retrieval enables both specific fact retrieval and broad thematic understanding.
"""
    },
    {
        "id": "sub_qa",
        "source": "Complex Question Answering via Sub-Question Decomposition — Perez et al. (2020)",
        "url": "https://arxiv.org/abs/2002.09758",
        "content": """
Complex questions can be decomposed into a sequence of simpler sub-questions that can
each be answered independently. The answers to sub-questions are then combined to answer
the original complex question.

Decomposition strategies:
  Sequential: answer sub-Q1, use answer in sub-Q2, chain results
  Parallel:   answer sub-Q1 and sub-Q2 independently, fuse answers
  Conditional: answer sub-Q2 only if sub-Q1 answer meets condition

In REFRAG, query decomposition assigns each sub-question a granularity hint:
  Fine-grained (proposition): specific numerical facts, entity attributes
  Medium (sentence): process descriptions, causal relationships
  Coarse (chunk): background context, broad comparisons

Sub-question decomposition benchmark results:
  HotpotQA (2-hop): decomposition 74.3% F1 vs direct 61.8%
  MuSiQue (4-hop): 48.2% vs 31.5%
  2WikiMultihopQA: 69.4% vs 54.7%

The key insight: each sub-question is simpler and can be answered with higher precision
than the complex parent question, especially when paired with fine-grained retrieval.
"""
    },
    {
        "id": "hyde",
        "source": "Hypothetical Document Embeddings (HyDE) — Gao et al. (2022)",
        "url": "https://arxiv.org/abs/2212.10496",
        "content": """
HyDE (Hypothetical Document Embeddings) addresses the query-document mismatch problem:
queries are typically short (5–15 words) while documents are long (100–500 words),
leading to embedding space misalignment.

HyDE approach:
  1. Given query q, generate a hypothetical document d' using an LLM ("write a passage
     that answers this question")
  2. Embed d' instead of q for retrieval
  3. d' is in the same linguistic register as real documents → better embedding match

The key insight: a hypothetical answer to "What BLEU score did the Transformer achieve?"
embeds much closer to real documents containing BLEU scores than the short query itself.

HyDE results:
  TREC DL2019 nDCG@10: 0.756 vs direct retrieval 0.702 (+7.7%)
  BEIR (zero-shot, avg): 0.479 vs 0.443 (+8.1%)
  FEVER (fact verification): 90.1% vs 87.2%
  MS MARCO (dev): 0.415 MRR@10 vs 0.393

HyDE is particularly effective for complex, technical queries where the query phrasing
differs substantially from how information is expressed in documents.
In REFRAG, HyDE is applied at the proposition level: the hypothetical proposition is
used as the search key rather than the raw question.
"""
    },
    {
        "id": "llm_reranker",
        "source": "LLM-based Re-ranking for Passage Retrieval — Ma et al. (2023)",
        "url": "https://arxiv.org/abs/2310.07554",
        "content": """
LLM-based re-rankers use a language model to score candidate passages for relevance
to a query, providing more nuanced relevance judgments than embedding cosine similarity.

Approaches:
  Pointwise: LLM rates each passage independently (1–10 scale or binary)
  Listwise:  LLM receives all passages and returns a ranked permutation
  Pairwise:  LLM compares pairs of passages and produces preference judgments

Listwise re-ranking (RankGPT, Sun et al. 2023): "Given query Q, rank these passages
from most to least relevant: [P1, P2, ... Pn]". The LLM outputs a permutation.

LLM re-ranker vs BM25 baseline on BEIR:
  TREC DL2019 nDCG@10: 0.832 (LLM-rerank) vs 0.703 (BM25) vs 0.756 (bi-encoder)
  TREC DL2020 nDCG@10: 0.824 vs 0.704 vs 0.742

In REFRAG, the re-ranker operates over the mixed pool of propositions, sentences,
and chunks retrieved across all sub-questions and retrieval rounds. This ensures
that the finest-grained, most precisely relevant units bubble to the top.
"""
    },
    {
        "id": "self_rag",
        "source": "Self-RAG: Learning to Retrieve, Generate and Critique — Asai et al. (2023)",
        "url": "https://arxiv.org/abs/2310.11511",
        "content": """
Self-RAG trains a single LM that uses special reflection tokens to decide when to retrieve,
whether retrieved passages are relevant, whether the output is supported, and whether
the response is useful.

Reflection tokens:
  [Retrieve=Yes/No]  — should the model retrieve for this span?
  [IsREL=relevant/irrelevant] — is the retrieved passage relevant?
  [IsSUP=supported/partial/unsupported] — is output supported by the passage?
  [IsUSE=1-5] — how useful is the overall response?

The Self-RAG paradigm informs REFRAG's coverage scoring: just as Self-RAG asks
"Is this claim supported?", REFRAG scores each sub-question: "Is this sub-question
answered by the retrieved propositions?" → if not, trigger re-retrieval.

Self-RAG (7B) vs baselines:
  PopQA: 54.9% vs ChatGPT 50.8%
  ASQA: 38.2 EM-recall vs 35.1 (standard RAG)
  FactScore (biography): 72.4% vs 60.1% (ChatGPT)
  ARC-Challenge: 67.3% vs 66.2%

Adaptive retrieval reduces hallucination by only retrieving when needed,
avoiding the noise introduced by irrelevant retrieved passages.
"""
    },
    {
        "id": "colbert",
        "source": "ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction — Khattab & Zaharia (2020)",
        "url": "https://arxiv.org/abs/2004.12832",
        "content": """
ColBERT introduces late interaction: both query and document are fully encoded by
BERT, producing multi-vector representations. Relevance is computed via MaxSim:
for each query token embedding, find the most similar document token embedding.

MaxSim score: score(q,d) = Σ_{qi ∈ E_q} max_{dj ∈ E_d} (qi · dj)

ColBERT preserves token-level granularity — it naturally aligns with REFRAG's
proposition-level retrieval, since both operate at finer-than-passage granularity.
ColBERT v2 adds residual compression, reducing index size 6–8× while maintaining accuracy.

MS MARCO Passage Ranking MRR@10: ColBERT v2 = 39.7% vs DPR = 31.7% vs BM25 = 18.4%.
BEIR zero-shot average: ColBERT v2 = 50.0% vs DPR = 37.9% vs BM25 = 43.0%.

The MaxSim operation makes ColBERT excellent at matching questions to passages that
contain the answer in a specific token-span, not necessarily stated as a full sentence.
This is ideal for fine-grained proposition retrieval where exact entity/value matches matter.
"""
    },
    {
        "id": "fid",
        "source": "Fusion-in-Decoder (FiD) for Open-Domain QA — Izacard & Grave (2021)",
        "url": "https://arxiv.org/abs/2007.01282",
        "content": """
FiD (Fusion-in-Decoder) processes each retrieved passage independently in the encoder,
then fuses all encoded representations jointly in the decoder. This allows attending
over all passages simultaneously at generation time without concatenating them in the input.

Architecture: given query q and retrieved passages p1...pk:
  Encoder: independently encodes each [q; pi] pair
  Decoder: cross-attends over all encoded representations simultaneously

FiD with 100 passages outperforms single-passage models on NaturalQuestions and TriviaQA.
The decoder effectively learns to fuse and reconcile information from multiple sources.

FiD-k results:
  NQ: FiD-large 51.4 EM vs RAG 44.5 vs T5 29.1
  TriviaQA: FiD-large 67.6 EM vs RAG 56.1 vs T5 29.8

In REFRAG's context: FiD's multi-passage fusion principle is applied at the proposition
level — instead of fusing k passages in the decoder, REFRAG fuses k propositions from
multiple granularities, assembling a fine-grained answer with precise attribution.
"""
    },
]

# ══════════════════════════════════════════════════════════════════════
# DEMO DOCUMENTS — 3 rich technical documents for proposition extraction
# ══════════════════════════════════════════════════════════════════════

DEMO_DOCUMENTS: List[Dict[str, str]] = [
    {
        "id": "transformer_deep_dive",
        "source": "The Transformer Architecture: A Technical Deep Dive",
        "url": "https://arxiv.org/abs/1706.03762",
        "content": """
The Transformer model, introduced in "Attention Is All You Need" (Vaswani et al., 2017),
completely eliminated recurrence and convolutions from sequence modeling. The base model
contains 65 million parameters and was trained on 4 NVIDIA P100 GPUs for 12 hours.

The encoder consists of 6 identical layers. Each encoder layer has two sub-layers:
a multi-head self-attention mechanism and a position-wise fully connected feed-forward
network. Each sub-layer uses a residual connection followed by layer normalization.

Multi-Head Attention (MHA) computes h=8 attention heads in parallel, each with
dimension d_k = d_v = d_model/h = 64. The formula is:
  MHA(Q,K,V) = Concat(head_1,...,head_h) · W^O
  head_i = Attention(Q·W_i^Q, K·W_i^K, V·W_i^V)
  Attention(Q,K,V) = softmax(Q·K^T / √d_k) · V

The scaling factor 1/√d_k prevents dot products from growing large in magnitude,
which would push the softmax into regions with extremely small gradients.

Position encodings use sinusoidal functions:
  PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
  PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
These allow the model to attend to relative positions and generalize to longer sequences.

The Transformer achieved 28.4 BLEU on the WMT 2014 English-to-German translation task,
surpassing all previously reported models including ensembles by more than 2 BLEU.
On English-to-French translation, it achieved 41.0 BLEU after training for 3.5 days.
The large Transformer model achieved 28.4 BLEU for EN→DE and 41.8 BLEU for EN→FR.

The feed-forward network in each layer has inner dimensionality d_ff = 2048 and uses
ReLU activation: FFN(x) = max(0, xW_1 + b_1)W_2 + b_2.

Training used the Adam optimizer with β_1=0.9, β_2=0.98, ε=1e-9.
Learning rate scheduling: warmup over 4000 steps then inverse square root decay.
Dropout rate of 0.1 was applied to the output of each sub-layer and the embeddings.
Label smoothing with ε_ls = 0.1 improved BLEU score despite hurting perplexity.

The decoder has an additional cross-attention sub-layer per layer, attending over
the encoder output. Masking ensures the decoder is auto-regressive.
"""
    },
    {
        "id": "chromadb_technical",
        "source": "ChromaDB: AI-Native Vector Database Technical Reference",
        "url": "https://docs.trychroma.com/",
        "content": """
ChromaDB is an open-source AI-native vector database designed for storing, querying,
and managing embedding vectors. It is built on top of hnswlib for approximate nearest
neighbor (ANN) search and SQLite for metadata storage.

The HNSW (Hierarchical Navigable Small World) algorithm provides O(log n) query
complexity for ANN search. Each node in the HNSW graph connects to M nearest neighbors.
Construction complexity is O(n · M · log n). Default M=16, ef_construction=100.

ChromaDB supports four distance metrics: cosine similarity (default for normalized
embeddings), L2 Euclidean distance, inner product, and L1 Manhattan distance.
Cosine similarity is computed as: cos(θ) = (A·B) / (|A||B|) where A and B are vectors.

Maximal Marginal Relevance (MMR) retrieval balances relevance and diversity:
  MMR = argmax_d [λ·sim(d,q) − (1−λ)·max_{d'∈S} sim(d,d')]
  where S is the set of already selected documents and λ controls the trade-off.
  Default λ=0.65 gives 65% weight to relevance and 35% to diversity.

Multi-tenancy is implemented via collections. Each collection has its own HNSW index
and can store arbitrary metadata with each vector. Metadata filtering uses a WHERE clause:
  collection.query(query_embeddings=[...], where={"doc_type": {"$eq": "proposition"}})

Persistent storage: ChromaDB uses DuckDB (pre-0.4) or SQLite (0.4+) for document
metadata and hnswlib for the vector index. The index is serialized to disk as .bin files.

ChromaDB's Python API integrates natively with LangChain via Chroma.from_documents().
Batch ingestion supports up to 41,666 documents per batch (default limit).
The embedding function can be any callable, with support for OpenAI, HuggingFace,
Cohere, and custom embedding models.

Performance: a 1M-vector collection with cosine similarity requires ~4 GB RAM.
Query latency at 1M vectors: ~2ms for cosine similarity with HNSW.
Index construction for 1M vectors: ~10 minutes on a single CPU core.
"""
    },
    {
        "id": "rag_systems_survey",
        "source": "RAG Systems: A Technical Survey of Retrieval-Augmented Generation",
        "url": "https://arxiv.org/abs/2312.10997",
        "content": """
Retrieval-Augmented Generation (RAG) systems combine a retrieval component (vector search
over a document corpus) with a generation component (LLM) to produce grounded,
attributable answers. The three core RAG paradigms are: naive RAG, advanced RAG, and
modular RAG.

Naive RAG: index documents as fixed-size chunks → retrieve top-k by embedding similarity
→ concatenate with query → LLM generates answer. Limitations: chunk boundary artifacts,
context window pressure, no query-aware chunking, poor precision for specific facts.

Advanced RAG improvements include: pre-retrieval query enhancement (HyDE, query expansion),
post-retrieval re-ranking (cross-encoders, LLM-based), better chunking (sliding window,
semantic chunking, hierarchical chunking), and iterative retrieval loops (FLARE).

Modular RAG introduces components that can be assembled like building blocks:
Query Router (decide which index to query), Query Transformer (expand or rewrite),
Retriever (dense, sparse, or hybrid), Re-ranker (cross-encoder or LLM), Generator.
This modularity is the basis for systems like REFRAG.

Fine-grained RAG addresses the precision problem: instead of retrieving 512-token chunks,
decompose documents into atomic propositions (15–35 words each) that can be individually
indexed and retrieved with high precision. A 512-token chunk might contain 15–20
propositions; fine-grained retrieval returns only the 1–3 most relevant.

Key RAG metrics:
  Faithfulness: fraction of answer claims supported by retrieved context
  Answer relevance: how well the answer addresses the question
  Context precision: fraction of retrieved context that is useful
  Context recall: fraction of ground-truth evidence retrieved

State-of-the-art RAG systems on RAGAS benchmark:
  Faithfulness: top systems achieve 0.93–0.96
  Answer relevance: 0.91–0.95
  Context precision: 0.78–0.88 (fine-grained > coarse)
  Context recall: 0.72–0.85

Fine-grained retrieval particularly improves context precision: retrieving one precise
proposition instead of a 512-token chunk eliminates ~95% of irrelevant tokens from context.
"""
    },
]

# ══════════════════════════════════════════════════════════════════════
# PRIMARY PDF
# ══════════════════════════════════════════════════════════════════════

PRIMARY_PDF = {
    "url":    "https://arxiv.org/pdf/2004.04906",
    "local":  "./dpr_paper.pdf",
    "source": "Dense Passage Retrieval — Karpukhin et al. (2020)",
    "url_ref":"https://arxiv.org/abs/2004.04906",
}

# All reference URLs
ALL_REFERENCES = [
    ("Dense Passage Retrieval (DPR)",            "https://arxiv.org/abs/2004.04906"),
    ("FLARE: Active Retrieval",                  "https://arxiv.org/abs/2305.06983"),
    ("Propositionizer (Chen et al., 2023)",      "https://arxiv.org/abs/2312.06648"),
    ("RAPTOR (Sarthi et al., 2024)",             "https://arxiv.org/abs/2401.18059"),
    ("Sub-Question Decomposition",               "https://arxiv.org/abs/2002.09758"),
    ("HyDE (Gao et al., 2022)",                  "https://arxiv.org/abs/2212.10496"),
    ("LLM Re-ranking (Ma et al., 2023)",         "https://arxiv.org/abs/2310.07554"),
    ("Self-RAG (Asai et al., 2023)",             "https://arxiv.org/abs/2310.11511"),
    ("ColBERT (Khattab & Zaharia, 2020)",        "https://arxiv.org/abs/2004.12832"),
    ("Fusion-in-Decoder (Izacard & Grave, 2021)","https://arxiv.org/abs/2007.01282"),
]


In [9]:
"""
══════════════════════════════════════════════════════════
Document ingestion pipeline that builds three parallel indexes at
different granularities from the same source documents.

Pipeline per document:
  ① ChunkSplitter        → paragraph-level ChunkUnits (300–700 chars)
  ② SentenceSplitter     → sentence-level SentenceUnits per chunk
  ③ PropositionExtractor → atomic claim-level Propositions per sentence (LLM)
  ④ MultiGranularityIndex→ upsert all three levels into three ChromaDB collections

Parent-child linkage maintained throughout:
  Proposition.sent_id  → SentenceUnit.sent_id
  Proposition.chunk_id → ChunkUnit.chunk_id
  SentenceUnit.prop_ids← list of child Proposition IDs
  ChunkUnit.sent_ids   ← list of child SentenceUnit IDs

On proposition extraction failure (LLM unavailable / parse error):
  Falls back to rule-based splitting: one proposition ≈ one independent clause.

References:
  Propositionizer (Chen et al., 2023) https://arxiv.org/abs/2312.06648
  DPR (Karpukhin et al., 2020)        https://arxiv.org/abs/2004.04906
"""

'\n══════════════════════════════════════════════════════════\nDocument ingestion pipeline that builds three parallel indexes at\ndifferent granularities from the same source documents.\n\nPipeline per document:\n  ① ChunkSplitter        → paragraph-level ChunkUnits (300–700 chars)\n  ② SentenceSplitter     → sentence-level SentenceUnits per chunk\n  ③ PropositionExtractor → atomic claim-level Propositions per sentence (LLM)\n  ④ MultiGranularityIndex→ upsert all three levels into three ChromaDB collections\n\nParent-child linkage maintained throughout:\n  Proposition.sent_id  → SentenceUnit.sent_id\n  Proposition.chunk_id → ChunkUnit.chunk_id\n  SentenceUnit.prop_ids← list of child Proposition IDs\n  ChunkUnit.sent_ids   ← list of child SentenceUnit IDs\n\nOn proposition extraction failure (LLM unavailable / parse error):\n  Falls back to rule-based splitting: one proposition ≈ one independent clause.\n\nReferences:\n  Propositionizer (Chen et al., 2023) https://arxiv.org/abs/2312.066

In [10]:
from __future__ import annotations

import hashlib
import json
import re
import time
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple


In [11]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.runnables import RunnablePassthrough ,RunnableLambda
from sentence_transformers import CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

C:\Users\dhanu\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0527 22:32:29.609000 18528 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [12]:

# ══════════════════════════════════════════════════════════════════════
# PROMPT: Proposition extraction
# ══════════════════════════════════════════════════════════════════════

PROPOSITION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a proposition extractor. Decompose the given text into atomic propositions.\n"
     "Each proposition must be:\n"
     "  1. MINIMAL — exactly one factual claim\n"
     "  2. SELF-CONTAINED — interpretable without surrounding context\n"
     "     (expand pronouns/references: 'it' → actual name, 'this' → what it refers to)\n"
     "  3. ATTRIBUTABLE — a verifiable statement, not an opinion or question\n\n"
     "Guidelines:\n"
     "  • Keep propositions 10–50 words; merge very short ones\n"
     "  • Drop conjunctions ('and', 'also') that just connect — split instead\n"
     "  • Include entity context (e.g. 'The Transformer model's base variant uses 65M params')\n"
     "  • Numerical facts make the best propositions\n\n"
     "Respond ONLY with valid JSON (no markdown):\n"
     "{{\"propositions\": [\"...\", \"...\"], \"entity_spans\": [[\"ent1\", \"ent2\"], [...]]}}\n"
     "entity_spans[i] contains key entities mentioned in propositions[i]."),
    ("human", "Text to decompose:\n{text}"),
])



In [13]:

# ══════════════════════════════════════════════════════════════════════
# EMBEDDING MANAGER
# ══════════════════════════════════════════════════════════════════════

class EmbeddingManager:
    _instance: Optional["EmbeddingManager"] = None

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self._model: Optional[HuggingFaceEmbeddings] = None

    @classmethod
    def get(cls, cfg: Config) -> "EmbeddingManager":
        if cls._instance is None:
            cls._instance = EmbeddingManager(cfg)
        return cls._instance

    def init(self) -> "EmbeddingManager":
        if self._model is None:
            Log.step("Loading HuggingFace Embeddings (local, no API key needed)")
            Log.info(f"Model : {self.cfg.EMBEDDING_MODEL}")
            Log.info(f"Device: {self.cfg.EMBEDDING_DEVICE}")
            self._model = HuggingFaceEmbeddings(
                model_name=self.cfg.EMBEDDING_MODEL,
                model_kwargs={"device": self.cfg.EMBEDDING_DEVICE},
                encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
            )
            Log.ok("Embedding model ready (~90 MB, cached after first run)")
        return self

    @property
    def model(self) -> HuggingFaceEmbeddings:
        if self._model is None:
            self.init()
        return self._model


In [14]:

# ══════════════════════════════════════════════════════════════════════
# RULE-BASED FALLBACK PROPOSITION SPLITTER
# ══════════════════════════════════════════════════════════════════════

def _rule_based_propositions(text: str, min_len: int = 15) -> List[Tuple[str, List[str]]]:
    """
    Fallback proposition extraction using clause splitting.
    Splits on semicolons, colons, and conjunctions that typically
    introduce independent clauses: '; ', ', and ', ', but ', ', while '.
    Returns list of (proposition_text, []) — empty entity_spans.
    """
    # First split on sentence boundaries that aren't already in sentence list
    clauses: List[str] = []
    for sentence in re.split(r'(?<=[.!?])\s+', text.strip()):
        # Split on semicolons and clause-introducing conjunctions
        parts = re.split(r';\s+|,\s+(?:and|but|while|whereas|however)\s+', sentence)
        for part in parts:
            part = part.strip().rstrip('.,;:')
            if len(part) >= min_len and not part.endswith('?'):
                # Ensure starts with capital
                part = part[0].upper() + part[1:] if part else part
                clauses.append(part)

    # If we got nothing useful, return the full text as one proposition
    if not clauses:
        return [(text.strip(), [])]

    return [(c, []) for c in clauses]



In [15]:

# ══════════════════════════════════════════════════════════════════════
# SENTENCE SPLITTER
# ══════════════════════════════════════════════════════════════════════

def split_sentences(text: str, min_len: int = 20) -> List[str]:
    """
    Split text into sentences using regex boundary detection.
    Handles common abbreviations (Mr., Dr., e.g., i.e., etc.).
    Merges very short sentences with the preceding one.
    """
    # Protect common abbreviations
    protected = re.sub(
        r'\b(Mr|Mrs|Ms|Dr|Prof|Sr|Jr|vs|etc|e\.g|i\.e|Fig|Eq|al)\.',
        lambda m: m.group(0).replace('.', '<DOT>'),
        text,
    )
    # Split on sentence boundaries
    raw = re.split(r'(?<=[.!?])\s+(?=[A-Z])', protected)
    sentences: List[str] = []
    for s in raw:
        s = s.replace('<DOT>', '.').strip()
        if not s:
            continue
        if len(s) < min_len and sentences:
            # Merge short fragments with preceding sentence
            sentences[-1] = sentences[-1] + ' ' + s
        else:
            sentences.append(s)
    return [s for s in sentences if len(s) >= min_len]



In [16]:

# ══════════════════════════════════════════════════════════════════════
# PROPOSITION EXTRACTOR
# ══════════════════════════════════════════════════════════════════════

class PropositionExtractor:
    """
    Extracts atomic propositions from sentences using GPT-4 (with rule fallback).
    Each proposition is a self-contained, minimal, attributable claim.

    Based on: Chen et al. (2023) https://arxiv.org/abs/2312.06648
    """

    def __init__(self, cfg: Config, llm: Optional[AzureChatOpenAI] = None):
        self.cfg    = cfg
        self.llm    = llm
        self._parser = StrOutputParser()

    def extract(self, sentence_text: str) -> List[Tuple[str, List[str]]]:
        """
        Extract propositions from a sentence.
        Returns list of (proposition_text, entity_spans).
        Falls back to rule-based splitting if LLM unavailable or fails.
        """
        if self.llm is None:
            return _rule_based_propositions(sentence_text, self.cfg.MIN_PROP_LEN)

        try:
            chain  = PROPOSITION_PROMPT | self.llm | self._parser
            raw    = chain.invoke({"text": sentence_text})
            clean  = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            data   = json.loads(clean)
            props  = data.get("propositions", [])
            spans  = data.get("entity_spans", [])
            # Pad entity_spans if shorter than props list
            while len(spans) < len(props):
                spans.append([])
            result = [
                (p.strip(), spans[i])
                for i, p in enumerate(props)
                if p.strip() and len(p.strip()) >= self.cfg.MIN_PROP_LEN
            ]
            return result if result else _rule_based_propositions(
                sentence_text, self.cfg.MIN_PROP_LEN
            )
        except Exception as exc:
            Log.warn(f"Proposition LLM fallback ({type(exc).__name__}): rule-based")
            return _rule_based_propositions(sentence_text, self.cfg.MIN_PROP_LEN)

    def extract_batch(
        self, sentences: List[str]
    ) -> List[List[Tuple[str, List[str]]]]:
        """Extract propositions for a batch of sentences."""
        return [self.extract(s) for s in sentences]


In [17]:


# ══════════════════════════════════════════════════════════════════════
# MULTI-GRANULARITY INDEX
# ══════════════════════════════════════════════════════════════════════

class MultiGranularityIndex:
    """
    Three parallel ChromaDB collections for proposition / sentence / chunk level.
    All three collections are built from the same source documents.

    Each collection stores:
      doc_id, source, granularity, parent_id, page_num, position
    as metadata alongside the text content and embedding.

    Retrieval queries can target any single granularity or all three.
    """

    def __init__(self, cfg: Config, em: EmbeddingManager):
        self.cfg    = cfg
        self.em     = em
        self._client: Optional[chromadb.PersistentClient] = None

        # LangChain Chroma wrappers (for MMR retrieval)
        self._prop_store:  Optional[Chroma] = None
        self._sent_store:  Optional[Chroma] = None
        self._chunk_store: Optional[Chroma] = None

        # Raw ChromaDB collections (for metadata queries)
        self._prop_coll:   Optional[Any] = None
        self._sent_coll:   Optional[Any] = None
        self._chunk_coll:  Optional[Any] = None

    def init_empty(self) -> "MultiGranularityIndex":
        """Create empty collections (ready to receive documents)."""
        Path(self.cfg.CHROMA_DIR).mkdir(parents=True, exist_ok=True)
        self._client = chromadb.PersistentClient(
            path=self.cfg.CHROMA_DIR,
            settings=Settings(anonymized_telemetry=False),
        )
        self._prop_coll  = self._client.get_or_create_collection(
            self.cfg.PROP_COLLECTION,  metadata={"hnsw:space": "cosine"})
        self._sent_coll  = self._client.get_or_create_collection(
            self.cfg.SENT_COLLECTION,  metadata={"hnsw:space": "cosine"})
        self._chunk_coll = self._client.get_or_create_collection(
            self.cfg.CHUNK_COLLECTION, metadata={"hnsw:space": "cosine"})

        p = self._prop_coll.count()
        s = self._sent_coll.count()
        c = self._chunk_coll.count()
        Log.ok(f"Multi-granularity index: {p} props · {s} sents · {c} chunks")
        return self

    def _lc_store(self, collection_name: str) -> Chroma:
        """Get or create a LangChain Chroma wrapper around a collection."""
        return Chroma(
            client=self._client,
            collection_name=collection_name,
            embedding_function=self.em.model,
        )

    def add_propositions(self, props: List[Proposition]):
        if not props:
            return
        docs, metas, ids = [], [], []
        for p in props:
            docs.append(p.content)
            metas.append({
                "doc_id":      p.doc_id,
                "chunk_id":    p.chunk_id,
                "sent_id":     p.sent_id,
                "source":      p.source,
                "granularity": Granularity.PROPOSITION.value,
                "position":    p.position,
                "page_num":    p.page_num or -1,
                "confidence":  p.confidence,
                "entities":    json.dumps(p.entity_spans[:10]),
            })
            ids.append(p.prop_id)
        try:
            self._prop_coll.upsert(documents=docs, metadatas=metas, ids=ids)
        except Exception as exc:
            Log.warn(f"Prop upsert failed: {exc}")

    def add_sentences(self, sents: List[SentenceUnit]):
        if not sents:
            return
        docs, metas, ids = [], [], []
        for s in sents:
            docs.append(s.content)
            metas.append({
                "doc_id":      s.doc_id,
                "chunk_id":    s.chunk_id,
                "source":      s.source,
                "granularity": Granularity.SENTENCE.value,
                "position":    s.position,
                "page_num":    s.page_num or -1,
                "prop_count":  len(s.prop_ids),
            })
            ids.append(s.sent_id)
        try:
            self._sent_coll.upsert(documents=docs, metadatas=metas, ids=ids)
        except Exception as exc:
            Log.warn(f"Sent upsert failed: {exc}")

    def add_chunks(self, chunks: List[ChunkUnit]):
        if not chunks:
            return
        docs, metas, ids = [], [], []
        for c in chunks:
            docs.append(c.content)
            metas.append({
                "doc_id":      c.doc_id,
                "source":      c.source,
                "granularity": Granularity.CHUNK.value,
                "page_num":    c.page_num or -1,
                "sent_count":  len(c.sent_ids),
                "prop_count":  len(c.prop_ids),
            })
            ids.append(c.chunk_id)
        try:
            self._chunk_coll.upsert(documents=docs, metadatas=metas, ids=ids)
        except Exception as exc:
            Log.warn(f"Chunk upsert failed: {exc}")

    # ── Retrieval ──────────────────────────────────────────────────────

    def query_propositions(
        self, query: str, k: int, mmr: bool = True
    ) -> List[Tuple[str, float, Dict]]:
        """Returns (content, score, metadata) for top-k propositions."""
        return self._query_collection(self._prop_coll, query, k, mmr, self.cfg.PROP_COLLECTION)

    def query_sentences(
        self, query: str, k: int, mmr: bool = True
    ) -> List[Tuple[str, float, Dict]]:
        return self._query_collection(self._sent_coll, query, k, mmr, self.cfg.SENT_COLLECTION)

    def query_chunks(
        self, query: str, k: int, mmr: bool = False
    ) -> List[Tuple[str, float, Dict]]:
        return self._query_collection(self._chunk_coll, query, k, mmr, self.cfg.CHUNK_COLLECTION)

    def _query_collection(
        self,
        coll: Any,
        query: str,
        k: int,
        mmr: bool,
        collection_name: str,
    ) -> List[Tuple[str, float, Dict]]:
        n_total = coll.count()
        if n_total == 0:
            return []
        actual_k = min(k, n_total)

        if mmr:
            # MMR via LangChain wrapper
            lc = self._lc_store(collection_name)
            results = lc.max_marginal_relevance_search(
                query, k=actual_k,
                fetch_k=min(self.cfg.FETCH_K, n_total),
                lambda_mult=self.cfg.MMR_LAMBDA,
            )
            return [(doc.page_content, 0.85, doc.metadata) for doc in results]
        else:
            # Direct cosine similarity
            res = coll.query(query_texts=[query], n_results=actual_k)
            if not res["ids"] or not res["ids"][0]:
                return []
            out = []
            for i, (doc, dist) in enumerate(
                zip(res["documents"][0], res["distances"][0])
            ):
                meta = res["metadatas"][0][i] if res.get("metadatas") else {}
                sim  = max(0.0, 1.0 - float(dist))
                out.append((doc, sim, meta))
            return out

    def counts(self) -> Tuple[int, int, int]:
        return (
            self._prop_coll.count(),
            self._sent_coll.count(),
            self._chunk_coll.count(),
        )



In [18]:

# ══════════════════════════════════════════════════════════════════════
# DOCUMENT INGESTION PIPELINE
# ══════════════════════════════════════════════════════════════════════

class DocumentIngestionPipeline:
    """
    Ingests raw documents → chunks → sentences → propositions.
    Populates the MultiGranularityIndex.

    Step 1: Split document into chunks (RecursiveCharacterTextSplitter)
    Step 2: Split each chunk into sentences (regex)
    Step 3: Extract propositions from each sentence (LLM or rule-based)
    Step 4: Upsert all three granularities into ChromaDB
    """

    def __init__(
        self,
        cfg:       Config,
        index:     MultiGranularityIndex,
        extractor: PropositionExtractor,
    ):
        self.cfg       = cfg
        self.index     = index
        self.extractor = extractor
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=cfg.CHUNK_SIZE,
            chunk_overlap=cfg.CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " "],
        )

    def ingest_document(
        self,
        content:  str,
        doc_id:   str,
        source:   str,
        url:      str = "",
        page_num: Optional[int] = None,
    ) -> Tuple[List[ChunkUnit], List[SentenceUnit], List[Proposition]]:
        """
        Full ingestion pipeline for one document.
        Returns (chunks, sentences, propositions) built from it.
        """
        Log.gran(Granularity.CHUNK, f"Ingesting: {source[:55]}")

        # ── Step 1: Chunk ─────────────────────────────────────────────
        lc_docs = self._splitter.create_documents(
            [content], metadatas=[{"source": source, "url": url, "doc_id": doc_id}]
        )
        # Deduplicate chunks
        seen_hashes: set = set()
        all_chunks:  List[ChunkUnit] = []
        all_sents:   List[SentenceUnit] = []
        all_props:   List[Proposition]  = []

        for chunk_pos, lc_doc in enumerate(lc_docs):
            chunk_text = lc_doc.page_content.strip()
            if not chunk_text:
                continue
            h = hashlib.md5(chunk_text.encode()).hexdigest()[:8]
            if h in seen_hashes:
                continue
            seen_hashes.add(h)

            chunk_id  = f"chk_{doc_id[:6]}_{h}"
            chunk_unit = ChunkUnit(
                chunk_id=chunk_id,
                content=chunk_text,
                doc_id=doc_id,
                source=source,
                page_num=page_num,
                metadata={"url": url, "position": chunk_pos},
            )

            # ── Step 2: Sentences ─────────────────────────────────────
            sentences = split_sentences(chunk_text, self.cfg.MIN_SENTENCE_LEN)
            for sent_pos, sent_text in enumerate(sentences):
                sent_h  = hashlib.md5(sent_text.encode()).hexdigest()[:8]
                sent_id = f"snt_{doc_id[:6]}_{h}_{sent_pos}"
                sent_unit = SentenceUnit(
                    sent_id=sent_id,
                    content=sent_text,
                    doc_id=doc_id,
                    chunk_id=chunk_id,
                    source=source,
                    page_num=page_num,
                    position=sent_pos,
                )

                # ── Step 3: Propositions ──────────────────────────────
                prop_tuples = self.extractor.extract(sent_text)
                for prop_pos, (prop_text, entities) in enumerate(prop_tuples):
                    prop_h  = hashlib.md5(prop_text.encode()).hexdigest()[:8]
                    prop_id = f"prp_{doc_id[:6]}_{h}_{sent_pos}_{prop_pos}"
                    prop    = Proposition(
                        prop_id=prop_id,
                        content=prop_text,
                        doc_id=doc_id,
                        chunk_id=chunk_id,
                        sent_id=sent_id,
                        source=source,
                        page_num=page_num,
                        position=prop_pos,
                        confidence=1.0,
                        entity_spans=entities,
                    )
                    sent_unit.prop_ids.append(prop_id)
                    chunk_unit.prop_ids.append(prop_id)
                    all_props.append(prop)

                chunk_unit.sent_ids.append(sent_id)
                all_sents.append(sent_unit)

            all_chunks.append(chunk_unit)

        # ── Step 4: Index all three granularities ─────────────────────
        self.index.add_chunks(all_chunks)
        self.index.add_sentences(all_sents)
        self.index.add_propositions(all_props)

        Log.gran(Granularity.PROPOSITION,
                 f"  → {len(all_chunks)} chunks | {len(all_sents)} sents "
                 f"| {len(all_props)} props")
        return all_chunks, all_sents, all_props

    def ingest_corpus(
        self, docs: List[Dict[str, str]]
    ) -> Tuple[int, int, int]:
        """Ingest a list of document dicts from corpus.py."""
        total_c = total_s = total_p = 0
        for doc in docs:
            c, s, p = self.ingest_document(
                content  = doc["content"].strip(),
                doc_id   = doc["id"],
                source   = doc["source"],
                url      = doc.get("url", ""),
            )
            total_c += len(c)
            total_s += len(s)
            total_p += len(p)
        return total_c, total_s, total_p

    def try_pdf(self) -> bool:
        """Attempt to download and ingest the primary reference PDF."""
        try:
            import urllib.request
            from langchain_community.document_loaders import PyPDFLoader

            if not Path(PRIMARY_PDF["local"]).exists():
                Log.info(f"Downloading PDF: {PRIMARY_PDF['url']}")
                urllib.request.urlretrieve(PRIMARY_PDF["url"], PRIMARY_PDF["local"])
                Log.ok(f"PDF saved → {PRIMARY_PDF['local']}")

            pages = PyPDFLoader(PRIMARY_PDF["local"]).load()
            full_text = " ".join(p.page_content for p in pages if p.page_content.strip())
            self.ingest_document(
                content  = full_text,
                doc_id   = "dpr_pdf",
                source   = PRIMARY_PDF["source"],
                url      = PRIMARY_PDF["url_ref"],
            )
            Log.ok(f"PDF ingested: {len(pages)} pages → propositions extracted")
            return True
        except Exception as exc:
            Log.warn(f"PDF unavailable ({exc}), skipping")
            return False


In [19]:
"""
════════════════════════════════════════════════════════════════
The core REFRAG engine implementing five retrieval strategies,
each exploiting the multi-granularity index:

  1. SINGLE_SHOT    — one retrieval across all three granularities, fused
  2. ITERATIVE      — re-retrieve until coverage ≥ threshold or max rounds
  3. DECOMPOSED     — query → sub-questions → per-sub retrieval → fuse
  4. HYDE           — hypothetical proposition as retrieval key (Gao 2022)
  5. FULL_REFRAG    — decompose + HyDE + multi-gran + iterative + rerank

Coverage Scoring:
  For each sub-question (or the main query if no decomposition),
  coverage = max cosine similarity between the sub-question embedding
  and retrieved proposition embeddings.
  coverage ≥ 0.75 → FULL; 0.4–0.75 → PARTIAL; < 0.4 → SPARSE

Re-ranking:
  Top RERANK_TOP_N candidates from all granularities are re-ranked
  by an LLM listwise ranker (Ma et al., 2023) then top RERANK_FINAL_K
  passed to the generator.

Citation Engine:
  Every factual sentence in the generated answer is matched to the
  most similar retrieved proposition. Similarity ≥ CITATION_SIM_THRESHOLD
  → attached as an inline citation [source · page · prop].

LLM Prompts defined here:
  DECOMPOSE_PROMPT     — break query into sub-questions
  HYDE_PROP_PROMPT     — generate hypothetical proposition
  COVERAGE_PROMPT      — score how well context covers a sub-question
  RERANK_PROMPT        — listwise re-ranker
  ANSWER_PROMPT        — final generation with citations
  FOLLOWUP_PROMPT      — generate follow-up retrieval queries for gaps
"""

'\n════════════════════════════════════════════════════════════════\nThe core REFRAG engine implementing five retrieval strategies,\neach exploiting the multi-granularity index:\n\n  1. SINGLE_SHOT    — one retrieval across all three granularities, fused\n  2. ITERATIVE      — re-retrieve until coverage ≥ threshold or max rounds\n  3. DECOMPOSED     — query → sub-questions → per-sub retrieval → fuse\n  4. HYDE           — hypothetical proposition as retrieval key (Gao 2022)\n  5. FULL_REFRAG    — decompose + HyDE + multi-gran + iterative + rerank\n\nCoverage Scoring:\n  For each sub-question (or the main query if no decomposition),\n  coverage = max cosine similarity between the sub-question embedding\n  and retrieved proposition embeddings.\n  coverage ≥ 0.75 → FULL; 0.4–0.75 → PARTIAL; < 0.4 → SPARSE\n\nRe-ranking:\n  Top RERANK_TOP_N candidates from all granularities are re-ranked\n  by an LLM listwise ranker (Ma et al., 2023) then top RERANK_FINAL_K\n  passed to the generator.\n\nC

In [20]:
from __future__ import annotations

import json
import time
import uuid
from typing import Any, Dict, List, Optional, Set, Tuple


In [21]:

# ══════════════════════════════════════════════════════════════════════
# PROMPTS
# ══════════════════════════════════════════════════════════════════════

DECOMPOSE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Decompose this complex query into at most {max_sq} simpler sub-questions.\n"
     "For each sub-question, specify:\n"
     "  • focus: what specific aspect it targets (1–4 words)\n"
     "  • granularity_hint: 'proposition' for specific numbers/facts, "
     "'sentence' for processes/explanations, 'chunk' for broad context\n\n"
     "Respond ONLY with valid JSON:\n"
     "{{\"sub_questions\": [{{\"question\": \"...\", \"focus\": \"...\", "
     "\"granularity_hint\": \"proposition|sentence|chunk\"}}]}}"),
    ("human", "Complex query: {query}"),
])

HYDE_PROP_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Write a short, factual passage (2–4 sentences) that would DIRECTLY answer "
     "this query if it existed in a technical document.\n"
     "Write as if you are an authoritative technical source.\n"
     "Include specific numbers, names, or values if applicable.\n"
     "This passage will be used as a retrieval query — make it dense with key terms."),
    ("human", "Query: {query}"),
])

COVERAGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Score how completely the given context answers the sub-question.\n"
     "0.0 = not answered at all\n"
     "0.5 = partially answered (relevant but incomplete)\n"
     "1.0 = fully answered with specific supporting evidence\n\n"
     "Respond ONLY with JSON: {{\"score\": 0.0-1.0, \"gap\": \"what is still missing (empty if full)\"}}"),
    ("human",
     "Sub-question: {sub_question}\n\nContext:\n{context}\n\nCoverage score:"),
])

RERANK_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Re-rank these passages from most to least relevant to the query.\n"
     "Consider: direct relevance, specificity, factual precision.\n"
     "Respond ONLY with a JSON array of the passage indices in ranked order "
     "(most relevant first): [3, 1, 5, 2, 4, ...]"),
    ("human",
     "Query: {query}\n\n"
     "Passages (0-indexed):\n{passages}\n\n"
     "Output ranked indices:"),
])

FOLLOWUP_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Generate {n} targeted follow-up retrieval queries to fill the gaps "
     "in the current coverage.\n"
     "Each follow-up query should target a specific missing piece of information.\n"
     "Make queries precise and keyword-rich for vector search.\n"
     "Respond ONLY with JSON: {{\"queries\": [\"query1\", \"query2\", ...]}}"),
    ("human",
     "Original query: {original_query}\n"
     "Coverage gaps: {gaps}\n"
     "Already retrieved content summary: {retrieved_summary}"),
])

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a precise, citation-focused answering system using fine-grained RAG.\n\n"
     "You have access to retrieved content at three granularities:\n"
     "  PROPOSITIONS — atomic factual claims (highest precision)\n"
     "  SENTENCES    — full sentences with surrounding context\n"
     "  CHUNKS       — paragraph-level context\n\n"
     "Rules:\n"
     "1. Answer using ONLY information from the retrieved context.\n"
     "2. Prefer proposition-level evidence for specific facts and numbers.\n"
     "3. Cite each factual claim: append [Source · location · gran] after the claim.\n"
     "   Example: 'The model achieves 28.4 BLEU [Transformer · §3 · prop].'\n"
     "4. If sub-questions were used, address each one in order.\n"
     "5. Note any gaps where coverage was insufficient.\n"
     "6. Be precise — do not generalize beyond what the propositions support.\n\n"
     "Retrieved context (finest → coarsest):\n"
     "--- PROPOSITIONS ---\n{prop_context}\n\n"
     "--- SENTENCES ---\n{sent_context}\n\n"
     "--- CHUNKS ---\n{chunk_context}"),
    ("human", "{query}"),
])



In [22]:

# ══════════════════════════════════════════════════════════════════════
# COVERAGE SCORER (embedding-based, fast — LLM optional)
# ══════════════════════════════════════════════════════════════════════

class CoverageScorer:
    """
    Scores how well retrieved propositions cover a sub-question.
    Primary method: max cosine similarity between sub-question embedding
    and all retrieved proposition embeddings.
    Secondary method: LLM scoring (slower but more nuanced).
    """

    def __init__(self, em: EmbeddingManager, llm: Optional[Any] = None):
        self.em  = em
        self.llm = llm
        self._parser = StrOutputParser()

    def score_embedding(
        self,
        sub_question: str,
        propositions: List[str],
        threshold: float = 0.75,
    ) -> Tuple[float, CoverageStatus]:
        """
        Coverage = max cosine similarity between the sub-question and any proposition.
        Returns (score, CoverageStatus).
        """
        if not propositions:
            return 0.0, CoverageStatus.NONE

        q_emb  = self.em.model.embed_query(sub_question)
        p_embs = self.em.model.embed_documents(propositions)

        # Cosine similarity (embeddings are already L2-normalized)
        max_sim = 0.0
        for p_emb in p_embs:
            sim = sum(a * b for a, b in zip(q_emb, p_emb))
            if sim > max_sim:
                max_sim = sim

        status = (
            CoverageStatus.FULL    if max_sim >= threshold else
            CoverageStatus.PARTIAL if max_sim >= 0.40       else
            CoverageStatus.SPARSE  if max_sim >= 0.20       else
            CoverageStatus.NONE
        )
        return max_sim, status

    def score_llm(
        self,
        sub_question: str,
        context: str,
    ) -> Tuple[float, str]:
        """LLM-based coverage scoring. Returns (score, gap_description)."""
        if self.llm is None:
            return 0.5, ""
        try:
            chain = COVERAGE_PROMPT | self.llm | self._parser
            raw   = chain.invoke({"sub_question": sub_question, "context": context})
            clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            data  = json.loads(clean)
            return float(data.get("score", 0.5)), data.get("gap", "")
        except Exception:
            return 0.5, ""



In [23]:

# ══════════════════════════════════════════════════════════════════════
# REFRAG ENGINE
# ══════════════════════════════════════════════════════════════════════

class REFRAGEngine:
    """
    Retrieval-Enhanced Fine-Grained RAG Engine.

    Builds multi-granularity index, dispatches retrieval strategies,
    runs the re-retrieval loop, applies LLM re-ranking, and generates
    proposition-cited answers.
    """

    def __init__(self, cfg: Config):
        self.cfg     = cfg
        self._parser = StrOutputParser()

        self.em:      Optional[EmbeddingManager]      = None
        self.index:   Optional[MultiGranularityIndex] = None
        self.scorer:  Optional[CoverageScorer]        = None
        self.llm:     Optional[AzureChatOpenAI]       = None

    # ── Setup ──────────────────────────────────────────────────────────

    def setup(
        self,
        index: MultiGranularityIndex,
        em: EmbeddingManager,
        llm: Optional[AzureChatOpenAI] = None,
    ) -> "REFRAGEngine":
        self.em     = em
        self.index  = index
        self.llm    = llm
        self.scorer = CoverageScorer(em, llm)
        Log.ok("REFRAGEngine ready — all 5 strategies available")
        return self

    def _invoke(self, prompt: ChatPromptTemplate, **kwargs) -> str:
        if self.llm is None:
            return "[LLM not configured]"
        chain = prompt | self.llm | self._parser
        last_exc: Optional[Exception] = None
        for attempt in range(1, 4):
            try:
                return chain.invoke(kwargs)
            except Exception as exc:
                last_exc = exc
                msg = str(exc).lower()
                is_conn = any(token in msg for token in ["connection", "timeout", "temporarily unavailable", "service unavailable", "429", "rate limit"])
                if attempt < 3 and is_conn:
                    Log.warn(f"LLM transient error (attempt {attempt}/3): {exc}. Retrying…")
                    time.sleep(0.8 * attempt)
                    continue
                break
        Log.err(f"LLM: {last_exc}")
        return f"[LLM error: {last_exc}]"

    def _parse_json(self, raw: str, fallback: Any = None) -> Any:
        try:
            clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            return json.loads(clean)
        except json.JSONDecodeError:
            return fallback if fallback is not None else {}

    # ── Query Decomposition ────────────────────────────────────────────

    def decompose_query(self, query: str) -> List[SubQuestion]:
        """Break a complex query into sub-questions with granularity hints."""
        Log.section("QUERY DECOMPOSITION", C.DECOMP)
        raw  = self._invoke(DECOMPOSE_PROMPT, query=query,
                             max_sq=self.cfg.MAX_SUB_QUESTIONS)
        data = self._parse_json(raw, {"sub_questions": []})
        sqs  = data.get("sub_questions", [])

        if not sqs:
            # Fallback: treat the whole query as one sub-question
            return [SubQuestion(
                sq_id=str(uuid.uuid4())[:8], question=query,
                parent_query=query, focus="all", granularity_hint="proposition",
            )]

        result: List[SubQuestion] = []
        for i, sq in enumerate(sqs[:self.cfg.MAX_SUB_QUESTIONS]):
            sub = SubQuestion(
                sq_id=str(uuid.uuid4())[:8],
                question=sq.get("question", query),
                parent_query=query,
                focus=sq.get("focus", f"aspect_{i+1}"),
                granularity_hint=sq.get("granularity_hint", "proposition"),
            )
            result.append(sub)
            Log.subq(sub, i + 1)

        return result

    # ── HyDE: Hypothetical Proposition Embedding ───────────────────────

    def hyde_query(self, query: str) -> str:
        """
        Generate a hypothetical document passage and use it as the retrieval key.
        Gao et al. (2022) https://arxiv.org/abs/2212.10496
        """
        Log.section("HyDE — Hypothetical Document Embedding", C.HYDE)
        hypo = self._invoke(HYDE_PROP_PROMPT, query=query)
        if hypo.startswith("[LLM"):
            Log.warn("HyDE fallback — using original query")
            return query
        Log.info(f"Hypothetical passage: {hypo[:120]}…")
        return hypo

    # ── Multi-Granularity Retrieval ────────────────────────────────────

    def retrieve_multi_gran(
        self,
        query: str,
        round_num: int = 1,
        gran_hint: Optional[str] = None,
    ) -> Tuple[GranularityResult, GranularityResult, GranularityResult]:
        """
        Retrieve from all three granularities simultaneously.
        gran_hint biases k values: 'proposition' → more props, etc.
        """
        k_prop  = self.cfg.TOP_K_PROP
        k_sent  = self.cfg.TOP_K_SENT
        k_chunk = self.cfg.TOP_K_CHUNK

        # Bias toward hinted granularity
        if gran_hint == Granularity.PROPOSITION.value:
            k_prop += 4; k_sent = max(2, k_sent - 1); k_chunk = max(2, k_chunk - 1)
        elif gran_hint == Granularity.SENTENCE.value:
            k_sent += 2
        elif gran_hint == Granularity.CHUNK.value:
            k_chunk += 2; k_prop = max(4, k_prop - 2)

        Log.gran(Granularity.PROPOSITION,
                 f"[round {round_num}] '{query[:55]}' k={k_prop}")
        prop_hits  = self.index.query_propositions(query, k=k_prop)

        Log.gran(Granularity.SENTENCE,
                 f"[round {round_num}] '{query[:55]}' k={k_sent}")
        sent_hits  = self.index.query_sentences(query, k=k_sent)

        Log.gran(Granularity.CHUNK,
                 f"[round {round_num}] '{query[:55]}' k={k_chunk}")
        chunk_hits = self.index.query_chunks(query, k=k_chunk)

        def _to_objs(hits, gran, unit_cls):
            objs, scores = [], {}
            for content, score, meta in hits:
                uid = f"{gran.value[:3]}_{str(uuid.uuid4())[:8]}"
                # Build typed objects from raw hits
                if gran == Granularity.PROPOSITION:
                    obj = Proposition(
                        prop_id=f"prp_{str(uuid.uuid4())[:8]}",
                        content=content,
                        doc_id=meta.get("doc_id", ""),
                        chunk_id=meta.get("chunk_id", ""),
                        sent_id=meta.get("sent_id", ""),
                        source=meta.get("source", ""),
                        page_num=meta.get("page_num") or None,
                        position=int(meta.get("position", 0)),
                        confidence=float(meta.get("confidence", 1.0)),
                        entity_spans=json.loads(meta.get("entities", "[]")),
                    )
                    obj.prop_id = uid  # reassign for dedup
                elif gran == Granularity.SENTENCE:
                    obj = SentenceUnit(
                        sent_id=f"snt_{str(uuid.uuid4())[:8]}",
                        content=content,
                        doc_id=meta.get("doc_id", ""),
                        chunk_id=meta.get("chunk_id", ""),
                        source=meta.get("source", ""),
                        page_num=meta.get("page_num") or None,
                        position=int(meta.get("position", 0)),
                    )
                else:
                    obj = ChunkUnit(
                        chunk_id=f"chk_{str(uuid.uuid4())[:8]}",
                        content=content,
                        doc_id=meta.get("doc_id", ""),
                        source=meta.get("source", ""),
                        page_num=meta.get("page_num") or None,
                    )
                objs.append(obj)
                # Explicit ID field per granularity type (avoids __dataclass_fields__ fragility)
                obj_id = (obj.prop_id  if gran == Granularity.PROPOSITION else
                          obj.sent_id  if gran == Granularity.SENTENCE     else
                          obj.chunk_id)
                scores[obj_id] = score
            return objs, scores

        # Build GranularityResult objects
        prop_objs,  prop_scores  = _to_objs(prop_hits,  Granularity.PROPOSITION, Proposition)
        sent_objs,  sent_scores  = _to_objs(sent_hits,  Granularity.SENTENCE,    SentenceUnit)
        chunk_objs, chunk_scores = _to_objs(chunk_hits, Granularity.CHUNK,       ChunkUnit)

        gr_prop  = GranularityResult(Granularity.PROPOSITION, prop_objs,  prop_scores,  query, round_num)
        gr_sent  = GranularityResult(Granularity.SENTENCE,    sent_objs,  sent_scores,  query, round_num)
        gr_chunk = GranularityResult(Granularity.CHUNK,       chunk_objs, chunk_scores, query, round_num)

        return gr_prop, gr_sent, gr_chunk

    # ── Coverage Assessment ────────────────────────────────────────────

    def assess_coverage(
        self,
        sub_questions: List[SubQuestion],
        prop_results:  List[GranularityResult],
    ) -> Tuple[float, List[str]]:
        """
        Score overall coverage across sub-questions.
        Returns (overall_score, [gap_descriptions]).
        """
        all_props = []
        for gr in prop_results:
            all_props.extend(item.content for item in gr.items
                              if hasattr(item, "content"))

        if not sub_questions:
            return 0.5, []

        scores, gaps = [], []
        for sq in sub_questions:
            score, status = self.scorer.score_embedding(
                sq.question, all_props, self.cfg.COVERAGE_THRESHOLD
            )
            sq.coverage_score = score
            scores.append(score)
            if status in (CoverageStatus.SPARSE, CoverageStatus.NONE):
                gaps.append(sq.question)

        overall = sum(scores) / len(scores) if scores else 0.0
        return overall, gaps

    # ── Follow-up Query Generation ─────────────────────────────────────

    def generate_followup_queries(
        self,
        original_query: str,
        gaps: List[str],
        retrieved_summary: str,
        n: int = 2,
    ) -> List[str]:
        """Generate targeted follow-up retrieval queries for coverage gaps."""
        if not gaps:
            return []
        raw = self._invoke(
            FOLLOWUP_PROMPT,
            original_query=original_query,
            gaps="; ".join(gaps[:4]),
            retrieved_summary=retrieved_summary[:400],
            n=n,
        )
        data = self._parse_json(raw, {"queries": []})
        queries = data.get("queries", [])[:n]
        Log.info(f"Follow-up queries: {queries}")
        return queries

    # ── LLM Re-ranker ──────────────────────────────────────────────────

    def rerank(
        self,
        query: str,
        prop_results: List[GranularityResult],
        sent_results: List[GranularityResult],
        chunk_results: List[GranularityResult],
    ) -> List[Any]:
        """
        Listwise LLM re-ranking of the candidate pool.
        Ma et al. (2023) https://arxiv.org/abs/2310.07554
        Returns up to RERANK_FINAL_K items, finest granularity first.
        """
        Log.section("LLM RE-RANKING", C.RERANK)

        # Collect candidates (props first — finest granularity)
        candidates: List[Tuple[str, Any]] = []
        for gr in prop_results:
            for item in gr.items:
                if hasattr(item, "content"):
                    candidates.append((item.content, item))
        for gr in sent_results:
            for item in gr.items:
                if hasattr(item, "content"):
                    candidates.append((item.content, item))
        for gr in chunk_results:
            for item in gr.items:
                if hasattr(item, "content") and len(candidates) < self.cfg.RERANK_TOP_N:
                    candidates.append((item.content, item))

        # Deduplicate by content hash
        seen: Set[str] = set()
        deduped: List[Tuple[str, Any]] = []
        for content, item in candidates:
            h = hashlib.md5(content[:100].encode()).hexdigest()[:8]
            if h not in seen:
                seen.add(h)
                deduped.append((content, item))

        top_n = deduped[:self.cfg.RERANK_TOP_N]
        if len(top_n) <= 1:
            return [item for _, item in top_n[:self.cfg.RERANK_FINAL_K]]

        # Format passages for re-ranker
        passages_text = "\n".join(
            f"[{i}] {content[:200]}" for i, (content, _) in enumerate(top_n)
        )

        raw  = self._invoke(RERANK_PROMPT, query=query, passages=passages_text)
        data = self._parse_json(raw, None)

        if isinstance(data, list):
            ranked_indices = [int(x) for x in data if isinstance(x, (int, float))
                               and 0 <= int(x) < len(top_n)]
            # Add any missed indices at the end
            for i in range(len(top_n)):
                if i not in ranked_indices:
                    ranked_indices.append(i)
            reranked = [top_n[i][1] for i in ranked_indices[:self.cfg.RERANK_FINAL_K]]
        else:
            reranked = [item for _, item in top_n[:self.cfg.RERANK_FINAL_K]]

        Log.info(f"Re-ranked {len(top_n)} → top {len(reranked)} candidates")
        return reranked

    # ── Answer Generation with Proposition Citations ───────────────────

    def generate_answer(
        self,
        query: str,
        prop_results: List[GranularityResult],
        sent_results: List[GranularityResult],
        chunk_results: List[GranularityResult],
        sub_questions: Optional[List[SubQuestion]] = None,
    ) -> Tuple[str, List[Tuple[str, str]]]:
        """
        Generate a fine-grained cited answer.
        Returns (answer_text, [(claim_fragment, citation_str), ...]).
        """
        # Format each granularity for the prompt
        prop_ctx = "\n".join(
            f"• {item.to_citation()}: {item.content}"
            for gr in prop_results for item in gr.items[:10]
            if hasattr(item, "to_citation")
        ) or "(none)"

        sent_ctx = "\n".join(
            f"• {item.to_citation()}: {item.content[:200]}"
            for gr in sent_results for item in gr.items[:6]
            if hasattr(item, "to_citation")
        ) or "(none)"

        chunk_ctx = "\n".join(
            f"• {item.to_citation()}: {item.content[:350]}"
            for gr in chunk_results for item in gr.items[:4]
            if hasattr(item, "to_citation")
        ) or "(none)"

        # Prefix sub-question context
        sq_prefix = ""
        if sub_questions:
            sq_prefix = "Sub-questions to address:\n" + "\n".join(
                f"  {i+1}. {sq.question}" for i, sq in enumerate(sub_questions)
            ) + "\n\n"

        full_query = sq_prefix + query

        answer = self._invoke(
            ANSWER_PROMPT,
            query=full_query,
            prop_context=prop_ctx,
            sent_context=sent_ctx,
            chunk_context=chunk_ctx,
        )

        # Extract inline citations from the answer text
        import re
        citations = re.findall(r'(.{30,80}?)(\[[^\]]+·[^\]]+·[^\]]+\])', answer)
        return answer, [(claim.strip(), cite) for claim, cite in citations]

    # ══════════════════════════════════════════════════════════════════
    # STRATEGY 1: SINGLE SHOT
    # ══════════════════════════════════════════════════════════════════

    def single_shot(self, query: str) -> REFRAGResult:
        t0 = time.time()
        Log.section("SINGLE-SHOT Multi-Granularity Retrieval", C.CYAN)

        gr_p, gr_s, gr_c = self.retrieve_multi_gran(query, round_num=1)
        prop_r = [gr_p]; sent_r = [gr_s]; chunk_r = [gr_c]

        # Coverage
        sq = SubQuestion(sq_id="sq0", question=query, parent_query=query,
                         focus="all", granularity_hint="proposition")
        cov, gaps = self.assess_coverage([sq], prop_r)
        status = (CoverageStatus.FULL if cov >= self.cfg.COVERAGE_THRESHOLD else
                  CoverageStatus.PARTIAL if cov >= 0.4 else CoverageStatus.SPARSE)

        Log.coverage(cov, 1, status.value)

        answer, citations = self.generate_answer(query, prop_r, sent_r, chunk_r)

        return REFRAGResult(
            query=query, strategy=RetrievalStrategy.SINGLE_SHOT,
            sub_questions=[sq], prop_results=prop_r, sent_results=sent_r,
            chunk_results=chunk_r, answer=answer, citations=citations,
            coverage_history=[cov], final_coverage=cov, coverage_status=status.value,
            retrieval_rounds=1,
            total_props=sum(len(gr.items) for gr in prop_r),
            total_sents=sum(len(gr.items) for gr in sent_r),
            total_chunks=sum(len(gr.items) for gr in chunk_r),
            elapsed_sec=round(time.time() - t0, 2),
        )

    # ══════════════════════════════════════════════════════════════════
    # STRATEGY 2: ITERATIVE RE-RETRIEVAL
    # ══════════════════════════════════════════════════════════════════

    def iterative(self, query: str) -> REFRAGResult:
        t0 = time.time()
        Log.section("ITERATIVE Re-Retrieval (coverage-driven)", C.CYAN)

        prop_r: List[GranularityResult] = []
        sent_r: List[GranularityResult] = []
        chunk_r: List[GranularityResult] = []
        coverage_history: List[float] = []
        gaps: List[str] = []
        current_query = query
        sq = SubQuestion(sq_id="sq0", question=query, parent_query=query,
                         focus="all", granularity_hint="proposition")

        for round_n in range(1, self.cfg.MAX_RETRIEVAL_ROUNDS + 1):
            gr_p, gr_s, gr_c = self.retrieve_multi_gran(
                current_query, round_num=round_n
            )
            prop_r.append(gr_p); sent_r.append(gr_s); chunk_r.append(gr_c)

            cov, gaps = self.assess_coverage([sq], prop_r)
            coverage_history.append(cov)
            status = (CoverageStatus.FULL if cov >= self.cfg.COVERAGE_THRESHOLD else
                      CoverageStatus.PARTIAL if cov >= 0.4 else CoverageStatus.SPARSE)
            Log.coverage(cov, round_n, status.value)

            # Stop if coverage threshold met
            if cov >= self.cfg.COVERAGE_THRESHOLD:
                Log.ok(f"Coverage threshold reached at round {round_n} ({cov:.1%})")
                break

            # Stop if no meaningful gain
            if (round_n > 1 and
                    len(coverage_history) >= 2 and
                    coverage_history[-1] - coverage_history[-2] < self.cfg.MIN_COVERAGE_GAIN):
                Log.warn(f"Coverage gain too small ({coverage_history[-1]-coverage_history[-2]:.1%}), stopping")
                break

            # Generate follow-up queries for gaps
            if gaps and round_n < self.cfg.MAX_RETRIEVAL_ROUNDS:
                retrieved_summary = " ".join(
                    item.content[:80] for gr in prop_r for item in gr.items[:3]
                    if hasattr(item, "content")
                )
                followups = self.generate_followup_queries(
                    query, gaps, retrieved_summary, n=2
                )
                if followups:
                    current_query = followups[0]
                    Log.info(f"Follow-up query: {current_query[:70]}")
                else:
                    break

        final_cov = coverage_history[-1] if coverage_history else 0.0
        final_status = (CoverageStatus.FULL if final_cov >= self.cfg.COVERAGE_THRESHOLD else
                        CoverageStatus.PARTIAL if final_cov >= 0.4 else CoverageStatus.SPARSE)

        answer, citations = self.generate_answer(query, prop_r, sent_r, chunk_r)

        return REFRAGResult(
            query=query, strategy=RetrievalStrategy.ITERATIVE,
            sub_questions=[sq], prop_results=prop_r, sent_results=sent_r,
            chunk_results=chunk_r, answer=answer, citations=citations,
            coverage_history=coverage_history, final_coverage=final_cov,
            coverage_status=final_status.value,
            retrieval_rounds=len(coverage_history),
            total_props=sum(len(gr.items) for gr in prop_r),
            total_sents=sum(len(gr.items) for gr in sent_r),
            total_chunks=sum(len(gr.items) for gr in chunk_r),
            elapsed_sec=round(time.time() - t0, 2),
        )

    # ══════════════════════════════════════════════════════════════════
    # STRATEGY 3: DECOMPOSED
    # ══════════════════════════════════════════════════════════════════

    def decomposed(self, query: str) -> REFRAGResult:
        t0 = time.time()
        Log.section("DECOMPOSED Query → Sub-questions", C.DECOMP)

        sub_questions = self.decompose_query(query)
        prop_r: List[GranularityResult] = []
        sent_r: List[GranularityResult] = []
        chunk_r: List[GranularityResult] = []

        for i, sq in enumerate(sub_questions, 1):
            Log.step(f"Sub-question {i}/{len(sub_questions)}: {sq.question[:60]}")
            gr_p, gr_s, gr_c = self.retrieve_multi_gran(
                sq.question, round_num=1, gran_hint=sq.granularity_hint
            )
            gr_p.sub_query = sq.question
            gr_s.sub_query = sq.question
            gr_c.sub_query = sq.question
            prop_r.append(gr_p); sent_r.append(gr_s); chunk_r.append(gr_c)

        cov, gaps = self.assess_coverage(sub_questions, prop_r)
        status = (CoverageStatus.FULL if cov >= self.cfg.COVERAGE_THRESHOLD else
                  CoverageStatus.PARTIAL if cov >= 0.4 else CoverageStatus.SPARSE)
        Log.coverage(cov, 1, status.value)

        answer, citations = self.generate_answer(
            query, prop_r, sent_r, chunk_r, sub_questions=sub_questions
        )

        return REFRAGResult(
            query=query, strategy=RetrievalStrategy.DECOMPOSED,
            sub_questions=sub_questions, prop_results=prop_r,
            sent_results=sent_r, chunk_results=chunk_r,
            answer=answer, citations=citations,
            coverage_history=[cov], final_coverage=cov, coverage_status=status.value,
            retrieval_rounds=1,
            total_props=sum(len(gr.items) for gr in prop_r),
            total_sents=sum(len(gr.items) for gr in sent_r),
            total_chunks=sum(len(gr.items) for gr in chunk_r),
            elapsed_sec=round(time.time() - t0, 2),
        )

    # ══════════════════════════════════════════════════════════════════
    # STRATEGY 4: HyDE
    # ══════════════════════════════════════════════════════════════════

    def hyde(self, query: str) -> REFRAGResult:
        t0 = time.time()
        Log.section("HyDE — Hypothetical Document Embedding", C.HYDE)

        hypo_query = self.hyde_query(query)
        gr_p, gr_s, gr_c = self.retrieve_multi_gran(hypo_query, round_num=1)

        # Also do a direct retrieval and merge (dual-key search)
        gr_p2, gr_s2, gr_c2 = self.retrieve_multi_gran(query, round_num=2)
        prop_r  = [gr_p, gr_p2]
        sent_r  = [gr_s, gr_s2]
        chunk_r = [gr_c, gr_c2]

        sq = SubQuestion(sq_id="sq0", question=query, parent_query=query,
                         focus="all", granularity_hint="proposition")
        cov, gaps = self.assess_coverage([sq], prop_r)
        status = (CoverageStatus.FULL if cov >= self.cfg.COVERAGE_THRESHOLD else
                  CoverageStatus.PARTIAL if cov >= 0.4 else CoverageStatus.SPARSE)
        Log.coverage(cov, 1, f"hyde+direct — {status.value}")

        answer, citations = self.generate_answer(query, prop_r, sent_r, chunk_r)

        return REFRAGResult(
            query=query, strategy=RetrievalStrategy.HYDE,
            sub_questions=[sq], prop_results=prop_r, sent_results=sent_r,
            chunk_results=chunk_r, answer=answer, citations=citations,
            coverage_history=[cov], final_coverage=cov, coverage_status=status.value,
            retrieval_rounds=2,
            total_props=sum(len(gr.items) for gr in prop_r),
            total_sents=sum(len(gr.items) for gr in sent_r),
            total_chunks=sum(len(gr.items) for gr in chunk_r),
            elapsed_sec=round(time.time() - t0, 2),
        )

    # ══════════════════════════════════════════════════════════════════
    # STRATEGY 5: FULL REFRAG (Decompose + HyDE + Iterative + Rerank)
    # ══════════════════════════════════════════════════════════════════

    def full_refrag(self, query: str) -> REFRAGResult:
        t0 = time.time()
        Log.section(
            "FULL REFRAG — Decompose + HyDE + Multi-Gran + Iterative + Rerank",
            C.MAG,
        )

        # ── Phase 1: Decompose ─────────────────────────────────────────
        sub_questions = self.decompose_query(query)

        prop_r: List[GranularityResult] = []
        sent_r: List[GranularityResult] = []
        chunk_r: List[GranularityResult] = []
        coverage_history: List[float] = []
        all_gaps: List[str] = []

        # ── Phase 2: Per-sub-question HyDE + multi-gran retrieval ─────
        for i, sq in enumerate(sub_questions, 1):
            Log.step(f"Sub-Q {i}: {sq.question[:55]}")

            # HyDE for sub-question
            hypo = self.hyde_query(sq.question)

            # Round 1: HyDE-enhanced retrieval
            gr_p, gr_s, gr_c = self.retrieve_multi_gran(
                hypo, round_num=1, gran_hint=sq.granularity_hint
            )
            gr_p.sub_query = sq.question
            gr_s.sub_query = sq.question
            gr_c.sub_query = sq.question
            prop_r.append(gr_p); sent_r.append(gr_s); chunk_r.append(gr_c)

            # Round 2: Direct sub-question retrieval (dual-key)
            gr_p2, gr_s2, gr_c2 = self.retrieve_multi_gran(
                sq.question, round_num=2, gran_hint=sq.granularity_hint
            )
            gr_p2.sub_query = sq.question
            prop_r.append(gr_p2); sent_r.append(gr_s2); chunk_r.append(gr_c2)

        # ── Phase 3: Coverage assessment ──────────────────────────────
        cov, gaps = self.assess_coverage(sub_questions, prop_r)
        coverage_history.append(cov)
        Log.coverage(cov, 1, CoverageStatus.PARTIAL.value)
        all_gaps.extend(gaps)

        # ── Phase 4: Iterative fill for gaps ─────────────────────────
        for round_n in range(2, self.cfg.MAX_RETRIEVAL_ROUNDS + 1):
            if cov >= self.cfg.COVERAGE_THRESHOLD or not all_gaps:
                break

            retrieved_summary = " ".join(
                item.content[:80] for gr in prop_r for item in gr.items[:4]
                if hasattr(item, "content")
            )
            followups = self.generate_followup_queries(
                query, all_gaps, retrieved_summary, n=2
            )
            if not followups:
                break

            for fq in followups:
                gr_p, gr_s, gr_c = self.retrieve_multi_gran(fq, round_num=round_n)
                prop_r.append(gr_p); sent_r.append(gr_s); chunk_r.append(gr_c)

            cov, all_gaps = self.assess_coverage(sub_questions, prop_r)
            coverage_history.append(cov)
            status = (CoverageStatus.FULL if cov >= self.cfg.COVERAGE_THRESHOLD else
                      CoverageStatus.PARTIAL)
            Log.coverage(cov, round_n, status.value)

            if (len(coverage_history) >= 2 and
                    coverage_history[-1] - coverage_history[-2] < self.cfg.MIN_COVERAGE_GAIN):
                break

        # ── Phase 5: LLM Re-ranking ────────────────────────────────────
        reranked = self.rerank(query, prop_r, sent_r, chunk_r)
        Log.gran(Granularity.PROPOSITION, f"Re-ranked → {len(reranked)} top items")

        # ── Phase 6: Answer generation with citations ─────────────────
        final_cov    = coverage_history[-1] if coverage_history else 0.0
        final_status = (CoverageStatus.FULL if final_cov >= self.cfg.COVERAGE_THRESHOLD else
                        CoverageStatus.PARTIAL if final_cov >= 0.4 else CoverageStatus.SPARSE)

        answer, citations = self.generate_answer(
            query, prop_r, sent_r, chunk_r, sub_questions=sub_questions
        )

        return REFRAGResult(
            query=query, strategy=RetrievalStrategy.FULL_REFRAG,
            sub_questions=sub_questions, prop_results=prop_r,
            sent_results=sent_r, chunk_results=chunk_r, reranked=reranked,
            answer=answer, citations=citations,
            coverage_history=coverage_history,
            final_coverage=final_cov, coverage_status=final_status.value,
            retrieval_rounds=len(coverage_history),
            total_props=sum(len(gr.items) for gr in prop_r),
            total_sents=sum(len(gr.items) for gr in sent_r),
            total_chunks=sum(len(gr.items) for gr in chunk_r),
            elapsed_sec=round(time.time() - t0, 2),
        )

    # ── Main dispatch ──────────────────────────────────────────────────

    def query(
        self,
        question: str,
        strategy: RetrievalStrategy = RetrievalStrategy.FULL_REFRAG,
    ) -> REFRAGResult:
        Log.banner(
            f"REFRAG QUERY — {strategy.value.upper()}",
            f"'{question[:65]}'",
        )

        dispatch = {
            RetrievalStrategy.SINGLE_SHOT: self.single_shot,
            RetrievalStrategy.ITERATIVE:   self.iterative,
            RetrievalStrategy.DECOMPOSED:  self.decomposed,
            RetrievalStrategy.HYDE:        self.hyde,
            RetrievalStrategy.FULL_REFRAG: self.full_refrag,
        }
        fn = dispatch.get(strategy, self.full_refrag)
        result = fn(question)
        Log.answer_box(result)

        # Log top propositions
        Log.section("TOP RETRIEVED PROPOSITIONS", C.PROP)
        shown: Set[str] = set()
        for gr in result.prop_results:
            for item in gr.items[:3]:
                if hasattr(item, "content") and item.content not in shown:
                    shown.add(item.content)
                    Log.prop(item)
                    if len(shown) >= 6:
                        break
            if len(shown) >= 6:
                break

        return result


In [24]:
# REFRAG hotfix: robust citation extraction + fallback citation injection
import re
from typing import Any, List, Optional, Tuple

def _extract_inline_citations(answer: str) -> List[Tuple[str, str]]:
    pairs: List[Tuple[str, str]] = []
    pattern = re.compile(r'(.{12,140}?)(\[[^\[\]\n]{6,140}\])', re.DOTALL)
    for m in pattern.finditer(answer):
        claim = m.group(1).strip()
        cite = m.group(2).strip()
        if claim and any(sep in cite for sep in ("·", "|", "/")):
            pairs.append((claim, cite))
    return pairs

def _best_fallback_citation(
    prop_results: List[Any], sent_results: List[Any], chunk_results: List[Any]
) -> Optional[str]:
    for group in (prop_results, sent_results, chunk_results):
        for gr in group:
            for item in getattr(gr, "items", [])[:5]:
                if hasattr(item, "to_citation"):
                    try:
                        cite = item.to_citation()
                        if isinstance(cite, str) and cite.startswith("[") and cite.endswith("]"):
                            return cite
                    except Exception:
                        pass
    return None

def _generate_answer_with_citation_fallback(
    self,
    query: str,
    prop_results: List[Any],
    sent_results: List[Any],
    chunk_results: List[Any],
    sub_questions: Optional[List[Any]] = None,
) -> Tuple[str, List[Tuple[str, str]]]:
    prop_ctx = "\n".join(
        f"• {item.to_citation()}: {item.content}"
        for gr in prop_results for item in gr.items[:10]
        if hasattr(item, "to_citation")
    ) or "(none)"

    sent_ctx = "\n".join(
        f"• {item.to_citation()}: {item.content[:200]}"
        for gr in sent_results for item in gr.items[:6]
        if hasattr(item, "to_citation")
    ) or "(none)"

    chunk_ctx = "\n".join(
        f"• {item.to_citation()}: {item.content[:350]}"
        for gr in chunk_results for item in gr.items[:4]
        if hasattr(item, "to_citation")
    ) or "(none)"

    sq_prefix = ""
    if sub_questions:
        sq_prefix = "Sub-questions to address:\n" + "\n".join(
            f"  {i+1}. {sq.question}" for i, sq in enumerate(sub_questions)
        ) + "\n\n"

    full_query = sq_prefix + query
    answer = self._invoke(
        ANSWER_PROMPT,
        query=full_query,
        prop_context=prop_ctx,
        sent_context=sent_ctx,
        chunk_context=chunk_ctx,
    )

    citations = _extract_inline_citations(answer)

    if not citations:
        fallback = _best_fallback_citation(prop_results, sent_results, chunk_results)
        if fallback:
            parts = re.split(r'(?<=[.!?])\s+', answer.strip())
            patched: List[str] = []
            for part in parts:
                part = part.strip()
                if not part:
                    continue
                if re.search(r'\[[^\]]+\]', part):
                    patched.append(part)
                else:
                    patched.append(f"{part} {fallback}")
            answer = " ".join(patched)
            citations = _extract_inline_citations(answer)

    return answer, citations

REFRAGEngine.generate_answer = _generate_answer_with_citation_fallback
print("Applied REFRAG citation hotfix: robust extraction + fallback injection.")

Applied REFRAG citation hotfix: robust extraction + fallback injection.


In [25]:

W = 76


# ══════════════════════════════════════════════════════════════════════
# DEMO QUERIES — one per strategy, showcasing different strengths
# ══════════════════════════════════════════════════════════════════════

DEMO_QUERIES = [
    # 1. Single-shot — specific numerical fact (best served by propositions)
    (
        RetrievalStrategy.SINGLE_SHOT,
        "What BLEU score did the Transformer achieve on WMT 2014 English-to-German?",
        "Single specific numerical fact — proposition-level precision shines here",
    ),
    # 2. Iterative — moderately complex, needs coverage across multiple docs
    (
        RetrievalStrategy.ITERATIVE,
        "How does MMR retrieval balance relevance and diversity in ChromaDB, "
        "and what are the default parameter values?",
        "Multi-aspect fact question — iterative fill covers both aspects",
    ),
    # 3. Decomposed — complex multi-part question that benefits from sub-question splitting
    (
        RetrievalStrategy.DECOMPOSED,
        "Compare DPR and ColBERT in terms of architecture, index construction cost, "
        "and retrieval performance on MS MARCO.",
        "Multi-entity comparison — decomposed into architecture/cost/performance sub-Qs",
    ),
    # 4. HyDE — technical query where query phrasing differs from document style
    (
        RetrievalStrategy.HYDE,
        "What technique prevents the dot product in attention from becoming too large?",
        "HyDE generates a passage phrased like the documents — better embedding match",
    ),
    # 5. Full REFRAG — complex multi-hop question requiring all features
    (
        RetrievalStrategy.FULL_REFRAG,
        "How do fine-grained RAG systems improve citation precision compared to "
        "standard chunk-level retrieval, and what benchmark results demonstrate this?",
        "Full pipeline: decompose + HyDE + iterative + rerank + proposition citations",
    ),
]


In [26]:

# ══════════════════════════════════════════════════════════════════════
# SYSTEM BUILDER
# ══════════════════════════════════════════════════════════════════════

def build_system(cfg: Config) -> REFRAGEngine:
    from langchain_openai import AzureChatOpenAI
    import socket
    from urllib.parse import urlparse

    Log.banner(
        "REFRAG: Retrieval-Enhanced Fine-Grained RAG",
        "Proposition · Sentence · Chunk — Three Parallel Indexes",
    )

    # ── Embeddings ─────────────────────────────────────────────────────
    em = EmbeddingManager.get(cfg)
    em.init()

    # ── Multi-granularity index ─────────────────────────────────────────
    index = MultiGranularityIndex(cfg, em)
    index.init_empty()

    # ── LLM ────────────────────────────────────────────────────────────
    llm = None
    if cfg.is_configured():
        Log.step("Connecting to Azure OpenAI")
        endpoint = getattr(cfg, 'AI_FOUNDRY_PROJECT_ENDPOINT', '').strip()
        fallback_endpoint = os.getenv('AI_FOUNDRY_PROJECT_ENDPOINT_FALLBACK', 'https://idkopenai.services.ai.azure.com')
        try:
            host = urlparse(endpoint).hostname or ''
            socket.gethostbyname(host)
        except Exception:
            Log.warn(f"Endpoint DNS resolution failed for {endpoint}; using fallback {fallback_endpoint}")
            endpoint = fallback_endpoint
        llm = AzureChatOpenAI(
            azure_endpoint=endpoint,
            azure_deployment=getattr(cfg, 'AI_FOUNDRY_DEPLOYMENT_NAME', 'gpt-4o'),
            api_version=getattr(cfg, 'AI_FOUNDRY_API_VERSION', '2024-12-01-preview'),
            api_key=getattr(cfg, 'AI_FOUNDRY_API_KEY', ''),
            temperature=getattr(cfg, 'LLM_TEMPERATURE', 0.0),
            max_tokens=getattr(cfg, 'LLM_MAX_TOKENS', 512),
            timeout=45,
            max_retries=4,
        )
        Log.ok(f"LLM ready — {cfg.AI_FOUNDRY_DEPLOYMENT_NAME}")
    else:
        Log.warn("Azure OpenAI not configured — rule-based proposition extraction + no LLM generation")

    # ── Ingestion pipeline ─────────────────────────────────────────────
    extractor = PropositionExtractor(cfg, llm)
    pipeline  = DocumentIngestionPipeline(cfg, index, extractor)

    n_props, n_sents, n_chunks = index.counts()

    if n_props == 0:
        Log.step("Indexing documents across 3 granularities…")

        # Attempt primary PDF
        pipeline.try_pdf()

        # Demo documents (rich, multi-paragraph)
        Log.step("Ingesting DEMO_DOCUMENTS (3 technical docs)")
        c, s, p = pipeline.ingest_corpus(DEMO_DOCUMENTS)
        Log.ok(f"Demo docs: {c} chunks · {s} sentences · {p} propositions")

        # Knowledge base (10 reference papers)
        Log.step("Ingesting KB_DOCS (10 reference papers)")
        c2, s2, p2 = pipeline.ingest_corpus(KB_DOCS)
        Log.ok(f"KB docs: {c2} chunks · {s2} sentences · {p2} propositions")

        n_props, n_sents, n_chunks = index.counts()
    else:
        Log.ok(f"Existing index: {n_props} props · {n_sents} sents · {n_chunks} chunks")

    Log.section("INDEX READY", C.GREEN)
    Log.gran(
        Granularity.PROPOSITION,
        f"{n_props:,} atomic propositions indexed"
    )
    Log.gran(
        Granularity.SENTENCE,
        f"{n_sents:,} sentence units indexed"
    )
    Log.gran(
        Granularity.CHUNK,
        f"{n_chunks:,} paragraph chunks indexed"
    )

    # ── Print references ──────────────────────────────────────────────
    Log.step("Reference papers:")
    for i, (title, url) in enumerate(ALL_REFERENCES, 1):
        Log.info(f"  [{i:2d}] {title}")
        Log.info(f"        {url}")

    # ── REFRAG engine ─────────────────────────────────────────────────
    engine = REFRAGEngine(cfg)
    engine.setup(index, em, llm)
    return engine



In [27]:


# ══════════════════════════════════════════════════════════════════════
# RUN MODES
# ══════════════════════════════════════════════════════════════════════

def run_demo(engine: REFRAGEngine, n: int = 0):
    """Run all 5 demo queries (n=0) or a specific one (n=1–5)."""
    queries = DEMO_QUERIES if n == 0 else [DEMO_QUERIES[n - 1]]
    print(f"\n{C.BOLD}{C.CYAN}Running {len(queries)} REFRAG demo "
          f"quer{'y' if len(queries)==1 else 'ies'}…{C.RESET}")
    for i, (strategy, query, note) in enumerate(queries, 1 if n == 0 else n):
        print(f"\n{C.BOLD}{C.YELLOW}{'▓'*W}")
        print(f"  Demo {i}: {note}")
        print(f"{'▓'*W}{C.RESET}")
        try:
            engine.query(query, strategy=strategy)
        except Exception as exc:
            Log.err(f"Demo {i} failed: {exc}")
        time.sleep(0.3)


def run_interactive(engine: REFRAGEngine):
    strategy = RetrievalStrategy.FULL_REFRAG

    print(f"\n{C.BOLD}{C.CYAN}")
    print("╔════════════════════════════════════════════════════════════╗")
    print("║    REFRAG — Fine-Grained RAG  ·  Interactive CLI           ║")
    print("║  Commands:                                                  ║")
    print("║    'strategy <name>'  — set strategy:                       ║")
    print("║       single_shot | iterative | decomposed | hyde |         ║")
    print("║       full_refrag                                           ║")
    print("║    'stats'  — show index statistics                         ║")
    print("║    'refs'   — print reference papers                        ║")
    print("║    'demo N' — run demo query N (1–5)                        ║")
    print("║    'q' / 'quit' — exit                                      ║")
    print("╚════════════════════════════════════════════════════════════╝")
    print(C.RESET)

    while True:
        try:
            user = input(
                f"\n{C.BOLD}[{strategy.value}] Query:{C.RESET} "
            ).strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user:
            continue
        cmd = user.lower()

        if cmd in ("q", "quit", "exit"):
            break

        if cmd == "stats":
            p, s, c = engine.index.counts()
            Log.section("INDEX STATISTICS")
            Log.gran(Granularity.PROPOSITION, f"{p:,} propositions")
            Log.gran(Granularity.SENTENCE,    f"{s:,} sentences")
            Log.gran(Granularity.CHUNK,       f"{c:,} chunks")
            continue

        if cmd == "refs":
            for i, (t, u) in enumerate(ALL_REFERENCES, 1):
                print(f"  [{i}] {t} — {u}")
            continue

        if cmd.startswith("strategy "):
            name = user[9:].strip().lower()
            try:
                strategy = RetrievalStrategy(name)
                Log.ok(f"Strategy: {strategy.value}")
            except ValueError:
                Log.warn(f"Unknown strategy '{name}'. "
                          f"Valid: {[s.value for s in RetrievalStrategy]}")
            continue

        if cmd.startswith("demo"):
            parts = cmd.split()
            n = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0
            if n:
                run_demo(engine, n)
            else:
                for i, (strat, q, note) in enumerate(DEMO_QUERIES, 1):
                    print(f"  {i}. [{strat.value}] {q[:65]}…")
            continue

        try:
            engine.query(user, strategy=strategy)
        except Exception as exc:
            Log.err(f"Error: {exc}")



In [42]:
# Extra examples you can run after setup
extra_demo_cases = [
    (
        "single_shot",
        "What exact scaling factor is applied to the dot-product attention scores in Transformer, and why?",
    ),
    (
        "decomposed",
        "Compare DPR, ColBERT, and BM25 in retrieval architecture and indexing trade-offs.",
    ),
    (
        "full_refrag",
        "How does proposition-level indexing improve attribution quality compared to chunk-level retrieval?",
    ),
    (
        "iterative",
        "Explain how MMR in ChromaDB balances relevance and diversity, including the lambda trade-off equation.",
    ),
    (
        "hyde",
        "Which optimization hyperparameters and schedule were used in the original Transformer training setup?",
    ),
]

RUN_EXTRA_DEMOS = False
if RUN_EXTRA_DEMOS:
    cfg = Config()
    engine = build_system(cfg)
    for i, (strategy_name, question) in enumerate(extra_demo_cases, 1):
        strategy = RetrievalStrategy(strategy_name)
        print(f"\n[Extra Demo {i}] {strategy.value}: {question}")
        engine.query(question, strategy=strategy)
else:
    print("Set RUN_EXTRA_DEMOS = True to execute extra REFRAG examples.")

Set RUN_EXTRA_DEMOS = True to execute extra REFRAG examples.


In [36]:
# Verification runner for newly added REFRAG tests
RUN_REFRAG_VERIFICATION = True

if RUN_REFRAG_VERIFICATION:
    cfg = Config()
    engine = build_system(cfg)
    for i, (strategy_name, question) in enumerate(extra_demo_cases[:3], 1):
        strategy = RetrievalStrategy(strategy_name)
        print(f"\n[REFRAG Verification {i}] {strategy.value}: {question}")
        try:
            engine.query(question, strategy=strategy)
        except Exception as exc:
            print(f"Verification case {i} failed: {type(exc).__name__}: {exc}")
            raise


════════════════════════════════════════════════════════════════════════════
  REFRAG: Retrieval-Enhanced Fine-Grained RAG
  Proposition · Sentence · Chunk — Three Parallel Indexes
════════════════════════════════════════════════════════════════════════════
  ✓ Multi-granularity index: 807 props · 676 sents · 152 chunks

▶ Connecting to Azure OpenAI
  ✓ LLM ready — gpt-4o
  ✓ Existing index: 807 props · 676 sents · 152 chunks

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  INDEX READY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[PROPOSITION ] 807 atomic propositions indexed
[SENTENCE    ] 676 sentence units indexed
[CHUNK       ] 152 paragraph chunks indexed

▶ Reference papers:
  ·   [ 1] Dense Passage Retrieval (DPR)
  ·         https://arxiv.org/abs/2004.04906
  ·   [ 2] FLARE: Active Retrieval
  ·         https://arxiv.org/abs/2305.06983
  ·   [ 3] Propositionizer (Chen et al., 2023)
  ·         https://arxiv.org/abs

In [41]:
# Compact verification summary
cfg = Config()
engine = build_system(cfg)

quick_checks = [
    (RetrievalStrategy.SINGLE_SHOT, extra_demo_cases[0][1]),
    (RetrievalStrategy.DECOMPOSED, extra_demo_cases[1][1]),
]

for strategy, question in quick_checks:
    result = engine.query(question, strategy=strategy)
    print(
        f"PASS | {strategy.value} | coverage={result.final_coverage:.2f} | "
        f"citations={len(result.citations)} | answer_len={len(result.answer)}"
    )


════════════════════════════════════════════════════════════════════════════
  REFRAG: Retrieval-Enhanced Fine-Grained RAG
  Proposition · Sentence · Chunk — Three Parallel Indexes
════════════════════════════════════════════════════════════════════════════
  ✓ Multi-granularity index: 807 props · 676 sents · 152 chunks

▶ Connecting to Azure OpenAI
  ✓ LLM ready — gpt-4o
  ✓ Existing index: 807 props · 676 sents · 152 chunks

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  INDEX READY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[PROPOSITION ] 807 atomic propositions indexed
[SENTENCE    ] 676 sentence units indexed
[CHUNK       ] 152 paragraph chunks indexed

▶ Reference papers:
  ·   [ 1] Dense Passage Retrieval (DPR)
  ·         https://arxiv.org/abs/2004.04906
  ·   [ 2] FLARE: Active Retrieval
  ·         https://arxiv.org/abs/2305.06983
  ·   [ 3] Propositionizer (Chen et al., 2023)
  ·         https://arxiv.org/abs

In [38]:
# Silent verification (suppresses verbose REFRAG logs)
_silent_methods = ["banner", "section", "gran", "step", "ok", "warn", "err", "info", "coverage", "prop", "subq", "answer_box"]
_original_log_methods = {name: getattr(Log, name) for name in _silent_methods}
for name in _silent_methods:
    setattr(Log, name, staticmethod(lambda *args, **kwargs: None))

try:
    cfg = Config()
    engine = build_system(cfg)
    checks = [
        (RetrievalStrategy.SINGLE_SHOT, extra_demo_cases[0][1]),
        (RetrievalStrategy.ITERATIVE, extra_demo_cases[3][1]),
        (RetrievalStrategy.FULL_REFRAG, extra_demo_cases[2][1]),
    ]
    for strategy, question in checks:
        result = engine.query(question, strategy=strategy)
        print(
            f"PASS | {strategy.value} | coverage={result.final_coverage:.2f} | "
            f"citations={len(result.citations)} | answer_len={len(result.answer)}"
        )
finally:
    for name, fn in _original_log_methods.items():
        setattr(Log, name, fn)

PASS | single_shot | coverage=0.51 | citations=1 | answer_len=466
PASS | iterative | coverage=0.62 | citations=1 | answer_len=963
PASS | full_refrag | coverage=0.69 | citations=7 | answer_len=2483


In [39]:
# Focused LLM connectivity probe for REFRAG
cfg = Config()
engine = build_system(cfg)
probe = engine._invoke(HYDE_PROP_PROMPT, query="What is the Transformer attention scaling factor?")
print("Probe status:", "OK" if not probe.startswith("[LLM error:") else probe)
print("Probe sample:", probe[:180])


════════════════════════════════════════════════════════════════════════════
  REFRAG: Retrieval-Enhanced Fine-Grained RAG
  Proposition · Sentence · Chunk — Three Parallel Indexes
════════════════════════════════════════════════════════════════════════════
  ✓ Multi-granularity index: 807 props · 676 sents · 152 chunks

▶ Connecting to Azure OpenAI
  ✓ LLM ready — gpt-4o
  ✓ Existing index: 807 props · 676 sents · 152 chunks

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  INDEX READY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[PROPOSITION ] 807 atomic propositions indexed
[SENTENCE    ] 676 sentence units indexed
[CHUNK       ] 152 paragraph chunks indexed

▶ Reference papers:
  ·   [ 1] Dense Passage Retrieval (DPR)
  ·         https://arxiv.org/abs/2004.04906
  ·   [ 2] FLARE: Active Retrieval
  ·         https://arxiv.org/abs/2305.06983
  ·   [ 3] Propositionizer (Chen et al., 2023)
  ·         https://arxiv.org/abs

In [40]:
# Kernel-level Azure connectivity diagnostics (no secrets printed)
import os
import requests

cfg = Config()
endpoint = cfg.AI_FOUNDRY_PROJECT_ENDPOINT.rstrip('/')
print('Endpoint:', endpoint)
print('Deployment:', cfg.AI_FOUNDRY_DEPLOYMENT_NAME)
print('API key present:', bool(cfg.AI_FOUNDRY_API_KEY))
print('AI_FOUNDRY_API_KEY env present:', bool(os.getenv('AI_FOUNDRY_API_KEY')))
print('AZURE_OPENAI_API_KEY env present:', bool(os.getenv('AZURE_OPENAI_API_KEY')))

try:
    r = requests.get(endpoint, timeout=15)
    print('HTTP reachability:', r.status_code)
except Exception as exc:
    print('HTTP reachability error:', type(exc).__name__, exc)

Endpoint: https://idkopenai.services.ai.azure.com
Deployment: gpt-4o
API key present: True
AI_FOUNDRY_API_KEY env present: False
AZURE_OPENAI_API_KEY env present: False
HTTP reachability: 200
